# ARMD 从论文到代码：完全自包含入门教程

**论文**：Auto-Regressive Moving Diffusion Models for Time Series Forecasting
Gao et al., *AAAI-25*，arXiv:[2412.09328](https://arxiv.org/abs/2412.09328)

**本教程目标**：
- 把论文的 **20 个公式（Eq.1–20）** 和 **2 个算法** 一一对应到代码；
- 把仓库核心路径（`linear.py / armd.py / solver.py / lr_sch.py / real_datasets.py / build_dataloader.py / main.py`）的关键代码、standalone replica 和等价流程内嵌到 notebook；
- 面向**有 Python + 机器学习基础但不熟悉 PyTorch / DDPM** 的读者，逐行解释实现细节。

**阅读路径**：

| 章节 | 主题 | 包含公式 |
|------|------|----------|
| A | 环境 + Stock 数据加载可视化 | — |
| B | DDPM 背景与 ARMD 动机 | Eq.11–20 |
| C | 前向扩散：滑动过程 | Eq.1–3 |
| D | 反向去噪：Devolution 网络 | Eq.4–7 |
| E | 采样/预测过程 | Eq.8–10 |
| F | 代码实现（Beta 调度 / model_utils / Linear / ARMD） | 全部 |
| G | 训练支持（LR 调度 / Trainer / DataLoader） | Algorithm 1 |
| H | 训练、评估与可视化 | Algorithm 2 |
| I | main.py 等价代码 + 消融实验分析 | — |

> **如何快速上手**：先运行所有 Code Cell（不看解释），确认管道跑通，再回头逐节阅读。
> **快速测试模式**：找到 `QUICK_TEST = False`，改为 `True`，约 3 分钟完成全流程。

**Paper reference (Stock, Table 1, z-score)**：MSE = 0.235, MAE = 0.269


## 项目源码地图：从 `main.py` 到公式实现

在进入公式前，先把仓库调用链摊开。ARMD 的源码不是一个单文件模型，而是由 YAML 配置动态实例化后串到训练器和数据集里的。
下面这张表是阅读本教程和回到原仓库调试时的导航图。

| 阶段 | 原始仓库入口 | 关键对象 / 函数 | standalone 中的位置 | 作用 |
|---|---|---|---|---|
| 配置读取 | `main.py` → `load_yaml_config(args.config_path)` | `Config/stock_paper.yaml` | H-1 | 决定模型参数、Stock 数据切分、batch、loss、RevIN、采样步数 |
| 动态实例化 | `Utils/io_utils.py::instantiate_from_config` | `target` 字符串 → Python 类 | F/G/H | 原仓库按 YAML 的 `target` 创建 `ARMD`、`CustomDataset`、scheduler；standalone 改为直接构造 |
| 训练数据 | `Data/build_dataloader.py::build_dataloader` | `CustomDataset(period='train')` | A-3 / G-3 | 构造 192 长度滑动窗口，并返回训练 DataLoader |
| 测试数据 | `Data/build_dataloader.py::build_dataloader_cond` | `CustomDataset(period='test', predict_length=96)` | A-3 / G-3 / H-4 | 构造测试窗口和预测 mask；mask 后续不参与 `sample_forecast` 打分 |
| 模型主类 | `Models/autoregressive_diffusion/armd.py::ARMD` | `forward`, `_train_loss`, `q_sample`, `fast_sample` | F-4 / C / D / E | 负责 Eq.1–3、Eq.6–10、训练 loss、RevIN 和采样链 |
| Devolution 网络 | `Models/autoregressive_diffusion/linear.py::Linear` | `forward`, `self.w`, `self.w_dev` | F-3 / D | 负责 Eq.4–5，同时包含训练扰动和双 schedule 细节 |
| 训练器 | `engine/solver.py::Trainer` | `train`, `sample_forecast`, `EMA` | G-2 / H-2 / H-4 | 负责梯度累积、优化器、LR scheduler、EMA 权重和评估循环 |
| 评估入口 | `main.py` for `run in range(10)` | `trainer.sample_forecast(...)` | H-4 / I-1 / J-3 | 10 次采样平均；当前 `fast_sample` 确定性，所以多次结果理论上相同 |

**从命令到指标的最短调用链**：

```text
python main.py --config_path ./Config/stock_paper.yaml
  -> main.py: load_yaml_config
  -> instantiate_from_config(configs["model"])
       -> ARMD(...) -> Linear(...)
  -> build_dataloader(configs, args)
       -> CustomDataset(train) -> DataLoader
  -> Trainer(...).train()
       -> ARMD.forward -> q_sample -> Linear.forward -> _train_loss
  -> build_dataloader_cond(configs, args)
       -> CustomDataset(test, predict_length=96) -> DataLoader
  -> for run in range(10): Trainer.sample_forecast(...)
       -> ema.ema_model.generate_mts -> fast_sample -> MSE/MAE against x[:, 96:, :]
```

**standalone 改写边界**：

- 保留：Stock 数据窗口、70/10/20 split、ARMD/Linear 数学逻辑、Trainer 训练口径、EMA 推理、10 次评估协议。
- 改写：不再通过 YAML `target` 动态 import，而是在 notebook 中直接构造对象；DataLoader 直接使用已创建的 `train_ds/test_ds`；scheduler 直接构造。
- 不执行：`main.py` 原文只作为参考展示，避免 standalone 依赖仓库包路径。

这张源码地图和最后的 J-3 Algorithm 审计表互相对应：这里看“项目怎么串起来”，J-3 看“论文算法每一步落到哪一行源码”。


## 符号与变量对照表

在阅读代码时，下表帮助你把论文符号映射到 Python 变量名：

| 论文符号 | 含义 | 代码变量 | 所在文件 |
|---|---|---|---|
| $X_{-L+1:0}$ | 历史序列（长度 L） | `x[:, :96, :]` | solver.py |
| $X^0_{1:T}$ | 未来序列（初态） | `x_start[:, 96:, :]` / `target` | armd.py |
| $X^t_{1-t:T-t}$ | 第 t 步中间态 | `x`（q_sample 输出） | armd.py |
| $X^T_{-T+1:0}$ | 历史序列（终态） | `x[:, :96, :]` | armd.py |
| $\bar\alpha_t$ | 累积乘积 $\prod_{k=1}^t\alpha_k$ | `alphas_cumprod[t]` | armd.py |
| $z_t$ | 真实演化趋势 | `target_noise` | armd.py `_train_loss` |
| $\hat z(t,\theta)$ | 预测演化趋势 | `pred_noise` | armd.py `_train_loss` |
| $\hat X^0$ | 预测未来初态 | `x_start`（fast_sample 内） | armd.py |
| $W(t)$ | 可学习权重（初始化为 $\bar\alpha_t$） | `self.w[t[0]]` | linear.py |
| $D$ | Linear 距离估计 | `x_tmp` | linear.py |
| $R(\cdot)$ | Devolution 网络 | `Linear`（类） | linear.py |
| `t_code` | 代码中的时间步（≠ 论文 t） | `t` in `randint` | armd.py |
| 论文 $t$ | 实际滑动步数 | `index = t_code + 1` | armd.py `q_sample` |
| $T$ | 最大扩散步数（=预测长度） | `self.num_timesteps = 96` | armd.py |
| $b, c, d$ | Eq.5 超参 | 硬编码: b=2, c=−1, d=0.5 | linear.py |
| $\eta_t$ | 训练扰动系数（代码实际为单步 $1-\beta_t$，不是累积 $\bar\alpha_t$） | `self.w_dev[t[0]]` | linear.py |


---
## Part A：环境检测与数据加载

> 先把环境跑通，看见数据。


### A-1  环境检测


In [ ]:
import importlib, sys, os
from pathlib import Path

REQUIRED = {"torch":"torch","einops":"einops","numpy":"numpy","pandas":"pandas",
            "sklearn":"scikit-learn","tqdm":"tqdm","ema_pytorch":"ema-pytorch","matplotlib":"matplotlib"}
missing = [pkg for mod,pkg in REQUIRED.items() if not importlib.util.find_spec(mod)]
if missing:
    print(f"[!] 缺少: {missing}  请运行: pip install {' '.join(missing)}")
else:
    print("[OK] 所有依赖已就绪")

import torch
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}")
print(f"GPU 可用: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f"  -> {torch.cuda.get_device_name(0)}")
else:
    print("  (CPU 模式，训练会慢但结果正确)")

# 定位仓库根目录（向上查找 Data/datasets/）
def repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "Data" / "datasets").is_dir():
            return cand
    return here

REPO_ROOT = repo_root()
DATA_PATH = REPO_ROOT / "Data" / "datasets" / "stock_data.csv"
print(f"仓库根目录: {REPO_ROOT}")
print(f"股票数据: {DATA_PATH}  存在={DATA_PATH.exists()}")


### A-2  Stock 数据集说明

**Stock 数据集**（来自 Diffusion-TS）：
- 谷歌股票日线数据，2004–2019，**3685 行**
- **6 个特征**：Open / High / Low / Close / Adj\_Close / Volume
- 无日期列（这与 ETTh 格式不同，需用 `name='stock'`）

**预处理流程**（对应 `Utils/Data_utils/real_datasets.py::CustomDataset`）：

```
原始数据 (3685, 6)
    ↓ StandardScaler.fit(全部行) → z-score 归一化
归一化数据 (3685, 6)
    ↓ 滑动窗口，步长1，窗口长度192
所有窗口 (3494, 192, 6)
    ↓ 时间顺序 70/10/20 切分（论文补充材料设置）
训练集 / 验证集 / 测试集
```

**为什么窗口长度 192？**  历史 96 步 + 未来 96 步 = 一个 ARMD 样本。

**为什么 z-score 归一化？** 论文 Table 1 的 MSE/MAE 都在归一化空间计算，不做反变换。

> **注意**：若真实 CSV 缺失，下方会自动生成随机游走占位数据，管道可跑通但指标不可与论文对比。


### A-3  `CustomDataset`（standalone replica）

下方是 `Utils/Data_utils/real_datasets.py::CustomDataset` 的 **standalone replica**：
保留 Stock 复现所需的窗口构造、z-score 归一化、70/10/20 时间顺序切分、预测 mask；
删掉或简化本教程不使用的通用功能，避免依赖仓库内部工具后无法单文件运行。

| 项目 | 原始仓库 | standalone 中的处理 |
|---|---|---|
| CSV 读取与 `name='stock'` | 保留 | 保留；仅 `name='etth'` 时丢弃日期列 |
| `StandardScaler.fit(data)` | 保留 | 保留；与 `norm_on_train=False` 的论文 Stock 配置一致 |
| `three_split=True` | 保留 | 保留；按窗口 70/10/20 时间顺序切分 |
| `neg_one_to_one` | 原代码参数存在，但当前源码里 `self.auto_norm=False` | 显式固定为 False，和当前源码行为一致 |
| `missing_ratio` 随机 mask | 调 `Utils.masking_utils.noise_mask` | 不实现；本教程只做预测，使用最后 `predict_length` 步 mask |
| `.npy` 保存 | 原代码会保存多种 ground truth / norm truth | 本教程不依赖保存文件，保留内存中的 `samples` |
| fMRI / ETTh 泛化路径 | 原文件还包含其它数据集逻辑 | 不纳入；standalone 专注 Stock 复现 |

**PyTorch 基础提示**：
- `torch.utils.data.Dataset`：PyTorch 数据集基类，子类必须实现 `__len__` 和 `__getitem__`。
- `__getitem__(idx)` 返回一个样本；DataLoader 会把多个样本拼成 batch。
- `torch.from_numpy(arr).float()` 把 numpy array 转成 PyTorch float32 张量。

**关键方法说明**：

| 方法 | 功能 |
|------|------|
| `read_data` | 读 CSV，`StandardScaler.fit`，返回原始数据和 scaler |
| `__normalize` | 用 scaler.transform 做 z-score 归一化 |
| `__getsamples_three_split` | 论文口径：70/10/20 时间顺序切分 |
| `__getsamples` | 默认 80/20 切分 |
| `divide` | 按 ratio 切分 regular/irregular 两段 |
| `__getitem__` | 训练期返回 `(x,)`；测试期返回 `(x, mask)` |


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset


class CustomDataset(Dataset):
    """Standalone replica of Utils/Data_utils/real_datasets.py::CustomDataset.

    Only supports stock CSV (name='stock') for this tutorial.
    Removes dependency on masking_utils and model_utils.
    """

    def __init__(
        self,
        name,
        data_root,
        window=192,
        proportion=0.8,
        save2npy=False,
        neg_one_to_one=False,
        seed=123,
        period="train",
        output_dir="./OUTPUT",
        predict_length=None,
        missing_ratio=None,
        style="separate",
        distribution="geometric",
        mean_mask_length=3,
        three_split=False,
        train_ratio=0.7,
        val_ratio=0.1,
    ):
        super().__init__()
        self.three_split  = three_split
        self.train_ratio  = float(train_ratio)
        self.val_ratio    = float(val_ratio)
        if self.three_split:
            assert period in ("train", "val", "test")
        else:
            assert period in ("train", "test")
        if period == "train":
            assert predict_length is None and missing_ratio is None

        self.name       = name
        self.pred_len   = predict_length
        self.auto_norm  = False   # neg_one_to_one scaling disabled in standalone

        self.rawdata, self.scaler = self.read_data(data_root, name)
        self.dir = os.path.join(output_dir, "samples")
        os.makedirs(self.dir, exist_ok=True)

        self.window, self.period = window, period
        self.len, self.var_num = self.rawdata.shape
        self.sample_num_total  = max(self.len - window + 1, 0)
        self.save2npy = save2npy

        # z-score normalization
        self.data = self.scaler.transform(self.rawdata)

        # split
        if self.three_split:
            train_w, val_w, test_w = self.__getsamples_three_split(self.data, seed)
            self.samples = {"train": train_w, "val": val_w, "test": test_w}[period]
        else:
            train_w, test_w = self.__getsamples(self.data, proportion, seed)
            self.samples = train_w if period == "train" else test_w

        # build mask for test/val (future region masked)
        if period in ("test", "val"):
            if predict_length is not None:
                masks = np.ones(self.samples.shape, dtype=bool)
                masks[:, -predict_length:, :] = False
                self.masking = masks
            else:
                raise NotImplementedError("missing_ratio masking not used in this tutorial")

        self.sample_num = self.samples.shape[0]

    # ── window construction ────────────────────────────────────────────────
    def __getsamples(self, data, proportion, seed):
        """Original 2-way split (used in stock.yaml, proportion=0.8)."""
        n = self.sample_num_total
        x = np.stack([data[i : i + self.window] for i in range(n)])
        return self.divide(x, proportion, seed)

    def __getsamples_three_split(self, data, seed):
        """Chronological 70/10/20 split (paper supplemental setting)."""
        n = self.sample_num_total
        x = np.stack([data[i : i + self.window] for i in range(n)])
        t_end = int(np.ceil(n * self.train_ratio))
        v_end = int(np.ceil(n * (self.train_ratio + self.val_ratio)))
        return x[:t_end], x[t_end:v_end], x[v_end:]

    # ── normalization helpers ──────────────────────────────────────────────
    def unnormalize(self, sq):
        d = self.scaler.inverse_transform(sq.reshape(-1, self.var_num))
        return d.reshape(-1, self.window, self.var_num)

    # ── static helpers ─────────────────────────────────────────────────────
    @staticmethod
    def divide(data, ratio, seed=2023):
        """Split windows into regular (first ceil(ratio*N)) and irregular (rest)."""
        size = data.shape[0]
        st0 = np.random.get_state()
        np.random.seed(seed)
        cut = int(np.ceil(size * ratio))
        idx = np.arange(size)          # chronological order (no shuffle)
        regular   = data[idx[:cut]]
        irregular = data[idx[cut:]]
        np.random.set_state(st0)
        return regular, irregular

    @staticmethod
    def read_data(filepath, name="stock"):
        """Read CSV; drop first column only for 'etth' format (has a date string)."""
        df = pd.read_csv(filepath, header=0)
        if name == "etth":
            df.drop(df.columns[0], axis=1, inplace=True)  # drop date column
        data = df.values.astype(np.float64)
        scaler = StandardScaler()
        scaler.fit(data)         # fit on ALL rows before splitting
        return data, scaler

    # ── PyTorch Dataset interface ──────────────────────────────────────────
    def __getitem__(self, ind):
        x = self.samples[ind]             # (window, var_num)  numpy float64
        x_t = torch.from_numpy(x).float()  # → PyTorch float32 tensor
        if self.period in ("test", "val"):
            m = self.masking[ind]
            return x_t, torch.from_numpy(m)
        return x_t

    def __len__(self):
        return self.sample_num


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# ── generate synthetic fallback if CSV missing ─────────────────────────────
def ensure_csv(path):
    if path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    z = np.cumsum(rng.standard_normal((900, 6)), axis=0)
    pd.DataFrame(z, columns=["Open","High","Low","Close","Adj_Close","Volume"])      .to_csv(path, index=False)
    print(f"[合成数据] {path} 已生成（占位，真实数据结果会不同）")
    return True

is_synth = ensure_csv(DATA_PATH)
print(f"使用{'合成' if is_synth else '真实'} CSV: {DATA_PATH}")

SEQ_LEN = 96    # 历史 = 预测 = 96 步（论文实验设置）
WINDOW  = 192   # 滑动窗口长度 = SEQ_LEN * 2

# ── 训练集（三段切分，对应 stock_paper.yaml） ──────────────────────────────
train_ds = CustomDataset(
    name="stock",
    data_root=str(DATA_PATH),
    window=WINDOW,
    three_split=True, train_ratio=0.7, val_ratio=0.1,
    period="train",
    save2npy=False,
)
# ── 测试集 ─────────────────────────────────────────────────────────────────
test_ds = CustomDataset(
    name="stock",
    data_root=str(DATA_PATH),
    window=WINDOW,
    three_split=True, train_ratio=0.7, val_ratio=0.1,
    period="test",
    predict_length=SEQ_LEN,
    save2npy=False,
)
N_FEAT = train_ds.var_num
print(f"特征数: {N_FEAT}  训练窗口: {len(train_ds)}  测试窗口: {len(test_ds)}")
print(f"单窗口形状: {train_ds[0].shape}  (192步 × {N_FEAT}特征)")


### A-4  可视化数据


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# 原始时序（前 500 行）
df_raw = pd.read_csv(DATA_PATH)
for col in df_raw.columns[:3]:
    axes[0].plot(df_raw[col].values[:500], alpha=0.8, lw=0.8, label=col)
axes[0].set_title("Stock 原始数据（前 500 行，归一化前）")
axes[0].legend(loc="upper right"); axes[0].grid(alpha=0.3)

# 单个训练窗口结构：历史 | 未来
win = train_ds[0].numpy()   # (192, 6)
x_h = np.arange(SEQ_LEN); x_f = np.arange(SEQ_LEN, 2*SEQ_LEN)
for c in range(3):
    axes[1].plot(x_h, win[:SEQ_LEN, c], lw=0.9, alpha=0.8)
    axes[1].plot(x_f, win[SEQ_LEN:, c], lw=0.9, alpha=0.8, ls="--")
axes[1].axvline(SEQ_LEN-0.5, color="red", ls="--", lw=1.5)
axes[1].axvspan(0, SEQ_LEN, alpha=0.05, color="steelblue", label="历史（已知）")
axes[1].axvspan(SEQ_LEN, WINDOW, alpha=0.05, color="darkorange", label="未来（预测目标）")
axes[1].set_title(f"训练窗口结构：{SEQ_LEN} 历史 + {SEQ_LEN} 未来 = {WINDOW} 步")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Batch 示意形状: (128, 192, 6)")
print("  轴 0 = batch (128 个窗口)")
print("  轴 1 = time  (192 时间步: 0-95 历史, 96-191 未来)")
print("  轴 2 = feat  (6 个股票特征)")


---
## Part B：背景知识 —— DDPM 与 ARMD 动机（论文 Eq.11–20）

> 本章对应论文 **Preliminary** 节和 **Supplemental Materials** 中的 DDPM 推导。
> 如果你已熟悉 DDPM，可跳过 B-1/B-2，直接看 B-3（ARMD 动机）。


### B-1  扩散模型基础（DDPM）—— Eq.11–14

DDPM（Ho et al., 2020）的核心思想：
- **前向过程**：给数据 $X^0$ 逐步加噪，直到变成纯高斯噪声 $X^T \sim \mathcal{N}(0,I)$。
- **反向过程**：训练神经网络从噪声逐步去噪还原数据。

**Eq.11 — 单步前向过程**：

$$q(X^t \mid X^{t-1}) = \mathcal{N}\!\left(X^t;\;\sqrt{1-\beta_t}\,X^{t-1},\;\beta_t I\right) \tag{11}$$

其中 $\beta_t \in [0,1]$ 是预定义噪声方差（"beta schedule"）。

**Eq.12 — 边缘分布：从 $X^0$ 直接采样第 $t$ 步**（不需要逐步迭代）：

$$q(X^t \mid X^0) = \mathcal{N}\!\left(X^t;\;\sqrt{\bar\alpha_t}\,X^0,\;(1-\bar\alpha_t)I\right) \tag{12}$$

**Eq.13 — 累积乘积**：

$$\bar\alpha_t = \prod_{k=1}^{t}\alpha_k, \quad \alpha_t = 1 - \beta_t \tag{13}$$

**Eq.14 — 等价重参数化形式**（可直接计算，无需逐步）：

$$\boxed{X^t = \sqrt{\bar\alpha_t}\,X^0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \quad \varepsilon \sim \mathcal{N}(0,I)} \tag{14}$$

**Eq.15 — 反向过程**：

$$p_\theta(X^{t-1} \mid X^t) = \mathcal{N}\!\left(X^{t-1};\;\mu_\theta(X^t,t),\;\sigma_t^2 I\right) \tag{15}$$

**两种训练目标**：
- **噪声预测**：网络预测 $\varepsilon$，推导出 $\mu_\theta$。
- **数据预测**：网络直接预测 $X^0$，从而计算 $\mu_\theta$。
  ARMD 采用后者：预测未来初态 $\hat X^0$。


### B-2  条件 DDPM 用于时间序列预测 —— Eq.16–17

把 DDPM 用于 TSF 的常规做法（论文所批评的方向）：

**Eq.16 — 条件生成模型**：

$$p_\theta(X^{0:T}_{1:F} \mid c) = p_\theta(X^T_{1:F})\prod_{t=1}^T p_\theta(X^{t-1}_{1:F} \mid X^t_{1:F}, c) \tag{16}$$

其中 $X^T_{1:F} \sim \mathcal{N}(0,I)$（纯高斯噪声），$c = g(X^0_{-L+1:0})$ 是从历史序列提取的条件。

**Eq.17 — 条件单步去噪**：

$$p_\theta(X^{t-1}_{1:F} \mid X^t_{1:F}, c) = \mathcal{N}\!\left(X^{t-1}_{1:F};\;\mu_\theta(X^t_{1:F},t \mid c),\;\sigma_t^2 I\right) \tag{17}$$

**论文的批评**：
1. 初始状态 $X^T \sim \mathcal{N}(0,I)$ 与历史序列**毫无关系**，大量的反向去噪步骤是在"从头生成"，效率极低。
2. 中间状态（加噪后的数据）不反映时间序列的**连续演化规律**，与 TSF 目标错位。
3. 历史信息以条件 $c$ 的形式注入，增加了模型复杂度和训练难度。


### B-2b  Eq.11–17 在仓库里到底是什么地位？

这一组公式很容易误读：它们不是“standalone notebook 少写了代码”，而是论文先介绍 DDPM/条件 DDPM 的标准做法，
再解释 ARMD 为什么要改掉这些做法。对照仓库时要分清三类：

| 公式 | 在仓库中的状态 | 具体线索 |
|---|---|---|
| Eq.11 DDPM 单步高斯加噪 | **没有作为 ARMD 前向过程执行** | `ARMD.q_sample` 不加高斯噪声，只做确定性滑动切片；`betas` 只用于构造系数 |
| Eq.12/Eq.14 DDPM 从 $X^0$ 直接采样 $X^t$ | **被 ARMD 改写为 Eq.2/Eq.3 的代数分解** | `_train_loss` 里用 `target_noise = (x - target*alpha)/minus_alpha` 反解 $z_t$ |
| Eq.13 $\bar\alpha_t$ 累积乘积 | **真实使用** | `alphas = 1 - betas`; `alphas_cumprod = torch.cumprod(alphas, dim=0)`；后续注册为 buffers |
| Eq.15 DDPM 反向高斯步 | **保留了传统扩散代码路径，但论文 Stock 配置不用它** | `q_posterior` / `p_mean_variance` / `p_sample` 存在；`model.fast_sampling=True` 时走 `fast_sample` 的 Eq.8–10 |
| Eq.16/Eq.17 条件 DDPM | **作为被 ARMD 替代的 baseline 思路，不在当前模型中实现** | 没有条件编码器 `c=g(history)`；`generate_mts(x)` 直接把 `x[:, :96, :]` 当采样链起点 |

所以阅读顺序应当是：先用 Eq.11–17 理解“传统扩散为什么不适合 TSF”，再看 Eq.1–10 如何把
“从噪声生成未来”替换为“从历史状态 devolution 到未来状态”。


### B-3  ARMA 理论 → ARMD 动机 —— Eq.18–20

ARMD 的名字和设计灵感来自 **ARMA（Auto-Regressive Moving Average）**：

**Eq.18 — AR 成分**：

$$x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2} + \cdots + \phi_p x_{t-p} + \varepsilon_t \tag{18}$$

**Eq.19 — MA 成分**：

$$x_t = \mu + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \cdots + \theta_q \varepsilon_{t-q} + \varepsilon_t \tag{19}$$

**Eq.20 — ARMA 完整模型**：

$$x_t = \underbrace{\phi_1 x_{t-1} + \cdots + \phi_p x_{t-p}}_{\text{AR: 自回归项}} + \underbrace{\theta_1\varepsilon_{t-1} + \cdots + \theta_q\varepsilon_{t-q}}_{\text{MA: 移动平均项}} + \varepsilon_t \tag{20}$$

**ARMD 的对应关系**（论文 Supplemental）：

| ARMA 概念 | ARMD 对应 |
|---|---|
| 历史值 $x_{t-i}$ | 滑动中间态的"历史部分" |
| 扰动项 $\varepsilon_{t-j}$ | 演化趋势 $z^t$（从未来到历史的偏移） |
| AR 系数 $\phi_i$ | Linear 模块的权重 $W(t)$ |
| MA 系数 $\theta_j$ | Linear 模块的线性层参数 |

**代码地位**：Eq.18–20 不是仓库里单独执行的 ARMA 模块；源码不会显式估计
`phi_i` / `theta_j` 或调用传统 ARMA 预测器。它们的作用是解释为什么 ARMD 把
“历史滑动状态”与“演化趋势残差”结合起来。可执行实现仍然落在 Eq.1–10：
`q_sample` 的滑动、`target_noise`/`pred_noise` 的趋势残差，以及 `Linear.forward`
里的 $W(t)$ 混合。

**核心改变**：不再加高斯噪声，改用**滑动（Slide）**作为前向演化：
- 初态 = 未来序列 $X^0_{1:T}$
- 终态 = 历史序列 $X^T_{-T+1:0}$（推理时已知！）
- 中间态 = 时间轴上的过渡窗口


### B-4  Beta Schedule 可视化

Beta Schedule 决定了 $\bar\alpha_t$ 的变化曲线，进而影响：
- 训练时 loss 的加权（`loss_weight`）
- Linear 中 W(t) 的初始化
- 推理时的更新步长

下面可视化 linear 和 cosine 两种 schedule 的差异。


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch

def linear_beta_schedule(timesteps):
    """Linear schedule: beta 从 beta_start 线性增加到 beta_end。
    scale = 1000/timesteps 是为了使不同 T 值下的尺度一致。
    """
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end   = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float64)

def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine schedule (Nichol & Dhariwal, 2021).
    相比 linear schedule，cosine 在 t 较小时变化更平缓，有助于保留更多数据信息。
    s=0.008 是避免 t=0 时 beta 过小的小偏移。
    """
    steps = timesteps + 1
    x     = torch.linspace(0, timesteps, steps, dtype=torch.float64)
    # cos((x/T + s) / (1+s) * pi/2)^2，然后归一化
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]   # 使 alpha_bar_0 = 1
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

T = 96   # Stock 实验中 T = SEQ_LEN = 96

betas_lin = linear_beta_schedule(T)
betas_cos = cosine_beta_schedule(T)

alphas_lin = 1. - betas_lin
alphas_cos = 1. - betas_cos
abar_lin = torch.cumprod(alphas_lin, dim=0)
abar_cos = torch.cumprod(alphas_cos, dim=0)

t_range = np.arange(1, T+1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t_range, betas_lin.numpy(),  label="linear beta",  lw=1.5)
axes[0].plot(t_range, betas_cos.numpy(),  label="cosine beta",  lw=1.5)
axes[0].set_title(r"$\beta_t$ (噪声方差 schedule)"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t_range, abar_lin.numpy(),  label=r"linear $\bar\alpha_t$",  lw=1.5)
axes[1].plot(t_range, abar_cos.numpy(),  label=r"cosine $\bar\alpha_t$",  lw=1.5)
axes[1].set_title(r"$\bar\alpha_t = \prod_{k=1}^t\alpha_k$ (累积乘积)"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(t_range, np.sqrt(abar_lin.numpy()),          label=r"$\sqrt{\bar\alpha_t}$ (linear)", lw=1.5)
axes[2].plot(t_range, np.sqrt(1-abar_lin.numpy()),        label=r"$\sqrt{1-\bar\alpha_t}$ (linear)", lw=1.5, ls="--")
axes[2].set_title(r"Eq.14 系数: $\sqrt{\bar\alpha}$ vs $\sqrt{1-\bar\alpha}$")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle("Beta Schedule 对比（ARMD 使用 cosine schedule 初始化 ARMD buffers，linear 初始化 Linear.w）")
plt.tight_layout(); plt.show()

print("关键数值（t=1, T/4, T/2, T）:")
for t_idx in [0, T//4-1, T//2-1, T-1]:
    print(f"  t_code={t_idx} (论文t={t_idx+1:2d}): "
          f"abar_lin={abar_lin[t_idx]:.4f}  abar_cos={abar_cos[t_idx]:.4f}")


---
## Part C：ARMD 前向扩散过程（Evolution）—— Eq.1–3

> 核心思想：不加高斯噪声，改用**确定性滑动窗口**作为前向演化。


### C-1  窗口约定与符号

在 ARMD 中，"上标"表示**扩散状态**，"下标"表示**时间覆盖范围**：

| 符号 | 说明 | 代码中的切片 |
|---|---|---|
| $X^0_{1:T}$ | 初态 = **未来序列**（训练时已知） | `x_start[:, 96:, :]` |
| $X^T_{-T+1:0}$ | 终态 = **历史序列**（推理时已知） | `x[:, :96, :]` |
| $X^t_{1-t:T-t}$ | 第 $t$ 步中间态 | `q_sample` 的输出 |

**代码时间步偏移**（非常重要！）：

论文里 $t \in \{1, 2, \ldots, T\}$，但代码里 `randint(0, T)` 得到的 `t_code` $\in \{0,\ldots,T-1\}$。

在 `q_sample` 里有 `index = int(t[0]) + 1`，所以：

$$t_{\text{论文}} = t_{\text{code}} + 1$$

这个 +1 确保：
- `t_code=0` → `index=1` → 滑动 1 步（接近未来初态）
- `t_code=95` → `index=96=T` → 滑动 T 步（等于完整历史终态）


### C-2  Eq.1 —— 单步滑动

$$\boxed{X^t_{1-t:T-t} = \mathrm{Slide}\!\left(X^{t-1}_{2-t:T-t+1},\;1\right)} \tag{1}$$

`Slide(X, k)` 表示将序列窗口 $X$ 向历史方向移动 $k$ 步。

**含义**：从第 $t-1$ 步的中间态，向历史方向移动 1 步，得到第 $t$ 步中间态。

这是**确定性操作**，没有随机性（区别于 DDPM 加噪！）。

### C-3  Eq.2 —— t 步直接计算中间态

$$\boxed{X^t_{1-t:T-t} = \underbrace{\mathrm{Slide}(X^0_{1:T},\;t)}_{\text{（a）滑动定义}} = \underbrace{\sqrt{\bar\alpha_t}\,X^0_{1:T} + \sqrt{1-\bar\alpha_t}\,z_t}_{\text{（b）扩散式分解}}} \tag{2}$$

Eq.2 把同一个 $X^t$ 写成了**两种等价形式**，初学者最容易困惑的就是它们怎么对应到代码——下面逐一拆开。

#### （a）`q_sample` 只实现了"滑动定义"这一半

```python
def q_sample(self, x_start, t, noise=None):
    index = int(t[0]) + 1                          # 论文 t = t_code + 1
    x_middle = x_start[:, pred_len-index : -index, :]
    return x_middle
```

注意：**代码里压根没有出现 $\sqrt{\bar\alpha_t}$、$z_t$、也没用到 `noise` 参数**。`q_sample` 做的纯粹是按下标切一个窗口，这正是 $\mathrm{Slide}(X^0_{1:T},\,t)$——把"历史+未来"拼接序列 `x_start`（长度 192）向历史方向滑 `index` 步后，取出长度 96 的窗口。

以 `pred_len=96` 为例：
- `t_code=0, index=1`：切 `x_start[:, 95:191, :]`（含1步历史+95步未来）
- `t_code=47, index=48`：切 `x_start[:, 48:144, :]`（各48步混合）
- `t_code=95, index=96`：切 `x_start[:, 0:96, :]`（完整历史）

#### （b）"扩散式分解"那一半在代码里是怎么体现的？

关键点（也是仅看 `q_sample` 看不出来的地方）：**ARMD 从不用右边的公式 $\sqrt{\bar\alpha_t}X^0+\sqrt{1-\bar\alpha_t}z_t$ 去"生成" $X^t$。** 计算方向恰好和 DDPM 相反：

| | DDPM（Eq.14） | ARMD（Eq.2） |
|---|---|---|
| 第 1 步 | 先**采样**噪声 $\varepsilon\sim\mathcal N(0,I)$ | 先**滑动**得到 $X^t$（`q_sample`） |
| 第 2 步 | 再算 $X^t=\sqrt{\bar\alpha_t}X^0+\sqrt{1-\bar\alpha_t}\varepsilon$ | 再**反解**出 $z_t$（见 Eq.3） |
| 谁是已知量 | $\varepsilon$ 是输入，$X^t$ 是输出 | $X^t$ 是输入，$z_t$ 是输出 |

所以右半边 $\sqrt{\bar\alpha_t}X^0+\sqrt{1-\bar\alpha_t}z_t$ **不是一段计算 $X^t$ 的代码**，而是一个**恒等式约束**：我们把已经切好的 $X^t$ 强行拆成"已知未来初态 $X^0$ 的缩放" + "一段残差"，其中缩放系数 $\sqrt{\bar\alpha_t},\sqrt{1-\bar\alpha_t}$ 由 beta schedule 事先固定，剩下的残差就**定义**为 $z_t$（演化趋势）。换句话说：

$$z_t \;\overset{\text{定义}}{=}\; \frac{X^t - \sqrt{\bar\alpha_t}\,X^0}{\sqrt{1-\bar\alpha_t}} \;\;\Longrightarrow\;\; \sqrt{\bar\alpha_t}X^0+\sqrt{1-\bar\alpha_t}z_t \equiv X^t\ \text{（恒成立）}$$

这一步**在 `q_sample` 里没有，而是落在 `_train_loss` 里**（下一节 Eq.3 给出对应代码）。也就是说，Eq.2 的（b）形式 = `q_sample` 的切片输出（a） + `_train_loss` 里 $z_t$ 的定义，两段代码合起来才完整对应 Eq.2。

> **为什么要费劲写成（b）形式？** 因为网络要预测的不是 $X^t$（它能直接切出来），而是这段残差 $z_t$（Eq.7 的回归目标）。把 $X^t$ 分解成"已知部分 + 残差"，才能定义出一个有意义的、随 $t$ 归一化的学习目标 $z_t$，并复用 DDIM 的采样公式（Eq.8–10）。

### C-4  Eq.3 —— 真实演化趋势 $z_t$

把上面的"定义"整理成论文形式（分子分母同除以 $\sqrt{\bar\alpha_t}$）：

$$\boxed{z_t = \frac{X^t_{1-t:T-t} - \sqrt{\bar\alpha_t}\,X^0_{1:T}}{\sqrt{1-\bar\alpha_t}} = \frac{\sqrt{1/\bar\alpha_t}\,X^t_{1-t:T-t} - X^0_{1:T}}{\sqrt{1/\bar\alpha_t - 1}}} \tag{3}$$

这正是把 Eq.2（b）反解出 $z_t$ 的结果——所以 Eq.2 和 Eq.3 是**同一个等式的两种摆放**，而 `q_sample`（给 $X^t$）+ Eq.3（给 $z_t$）合在一起，才把 Eq.2 完整落实到代码。

在 `_train_loss` 中，这对应（这就是 Eq.2 的（b）半在代码里真正出现的位置）：

```python
# x      = q_sample 输出 = X^t_{1-t:T-t}   shape: (B, 96, 6)   ← Eq.2(a) 切片结果
# target = X^0_{1:T} = 真实未来             shape: (B, 96, 6)
# alpha       = sqrt(alpha_bar_t)           ← Eq.2(b) 的 √ᾱ_t
# minus_alpha = sqrt(1 - alpha_bar_t)       ← Eq.2(b) 的 √(1-ᾱ_t)

target_noise = (x - target * alpha) / minus_alpha     # ← 这一行就是 Eq.3 / Eq.2(b) 反解
# 证明它与 Eq.3 一致：令 a = sqrt(abar_t)，则
#   target_noise = (x - X^0 * a) / sqrt(1-abar)
# 反过来代回去就得到 Eq.2(b):
#   a*X^0 + sqrt(1-abar)*target_noise
#   = a*X^0 + sqrt(1-abar) * (x - a*X^0)/sqrt(1-abar)
#   = a*X^0 + (x - a*X^0) = x = X^t   ✓ （恒等式，对任意 X^0 都成立）
```

下一个代码单元用真实股票窗口做**数值验证**：用 Eq.3 从切片 $X^t$ 反解 $z_t$，再用 Eq.2(b) 重建，确认结果与原始切片逐元素相等。

### C-5  数值验证：Eq.2 的两种形式逐元素相等


In [ ]:
import torch

# 复用 ARMD 的 schedule 系数（cosine, T=96），独立复算一遍以便本节自包含
def _cosine_abar(timesteps=96, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float64)
    ac = torch.cos(((x / timesteps) + s) / (1 + s) * 3.141592653589793 * 0.5) ** 2
    ac = ac / ac[0]
    betas = torch.clip(1 - (ac[1:] / ac[:-1]), 0, 0.999)
    alphas = 1.0 - betas
    return torch.cumprod(alphas, dim=0)            # alphas_cumprod, shape (96,)

PRED_LEN = 96
abar = _cosine_abar(PRED_LEN)

win0 = torch.from_numpy(train_ds[0].numpy()).double()   # (192, 6) 一个训练窗口
X0   = win0[PRED_LEN:, :]                                # X^0_{1:T} 真实未来 (96, 6)

print(f"{'t_code':>6} {'index':>5} {'切片==Slide':>11} {'Eq2(b)重建误差':>15}")
for t_code in [0, 23, 47, 71, 95]:
    index = t_code + 1                                   # 论文 t = t_code+1

    # —— Eq.2(a)：q_sample 的滑动切片 —— 这就是 X^t
    Xt = win0[PRED_LEN-index : (-index if index>0 else None), :]   # (96, 6)

    # —— Eq.3：从 X^t 反解 z_t（= _train_loss 里的 target_noise）——
    a  = abar[t_code].sqrt()
    ma = (1 - abar[t_code]).sqrt()
    z_t = (Xt - X0 * a) / ma

    # —— Eq.2(b)：用 √ᾱ·X^0 + √(1-ᾱ)·z_t 重建，应当 == Xt ——
    Xt_rebuilt = a * X0 + ma * z_t
    err = (Xt_rebuilt - Xt).abs().max().item()

    same_as_slide = torch.allclose(Xt, win0[PRED_LEN-index:PRED_LEN-index+PRED_LEN, :])
    print(f"{t_code:>6} {index:>5} {str(same_as_slide):>11} {err:>15.2e}")

print("\n结论：")
print(" • Eq.2(a) 切片 = q_sample 的输出，纯下标操作，无系数、无随机数；")
print(" • Eq.3 用 √ᾱ、√(1-ᾱ) 把该切片反解为 z_t（_train_loss 的 target_noise）；")
print(" • Eq.2(b) 重建误差≈机器精度(1e-15) ⇒ 两种形式确为同一 X^t 的恒等改写。")


### C-6  交互可视化：`q_sample` 滑动过程


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

PRED_LEN = 96
win0 = train_ds[0].numpy()                         # (192, 6)
x_start_d = torch.from_numpy(win0).unsqueeze(0)   # (1, 192, 6)

t_vals = [0, 23, 47, 71, 95]
feat   = 0   # 只展示第 0 个特征

fig, axes = plt.subplots(1, len(t_vals), figsize=(4*len(t_vals), 3.5), sharey=False)
colors = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(t_vals)))

true_future = win0[PRED_LEN:, feat]
true_hist   = win0[:PRED_LEN, feat]

for i, t_code in enumerate(t_vals):
    index = t_code + 1           # q_sample 中的 index = t_code + 1
    start = PRED_LEN - index
    end   = start + PRED_LEN
    x_mid = win0[start:end, feat]

    ax = axes[i]
    ax.plot(x_mid, color=colors[i], lw=1.5)
    if i == 0:
        ax.plot(true_future, color="green",  ls="--", lw=0.8, alpha=0.5, label="真实未来 X^0")
        ax.plot(true_hist,   color="purple", ls="--", lw=0.8, alpha=0.5, label="真实历史 X^T")
        ax.legend(fontsize=7)
    ax.set_title(
        f"t_code={t_code} (论文t={t_code+1})\n"
        f"切片[{start}:{end}]\n"
        f"{'≈未来初态' if t_code==0 else ('=历史终态' if t_code==95 else '过渡状态')}",
        fontsize=9,
    )
    ax.grid(alpha=0.3)

fig.suptitle("Eq.(1)(2) q_sample：t 增大 → 窗口从未来初态滑向历史终态（确定性，无随机噪声）")
plt.tight_layout(); plt.show()

# 验证 t_code=95 = 完整历史
assert np.allclose(win0[0:96, feat], true_hist), "验证失败"
print("验证通过: t_code=95 的切片 == 完整历史段")
print("验证通过: 无随机性（纯切片操作，不依赖任何随机数生成器）")


---
## Part D：ARMD 反向去噪过程（Devolution）—— Eq.4–7

> 网络的任务：给定中间态 $X^t$，预测未来初态 $\hat X^0$，从而计算预测演化趋势 $\hat z$，优化 Eq.7 的 L1 loss。


### D-1  Eq.4 —— Linear 距离预测 D

$$\boxed{D = \mathrm{Linear}(X^t_{1-t:T-t})} \tag{4}$$

`Linear` 模块是 ARMD 的核心 devolution 网络 $R(\cdot)$。

对每个特征维度独立做时间轴上的线性映射（`nn.Linear(T, T)`）：

```python
# input_ shape: (B, 96, 6)
# 对时间轴做线性映射，需要把时间维放到最后
x_tmp = self.linear(input_.permute(0, 2, 1)).permute(0, 2, 1)
#               (B, 6, 96) → Linear(96,96) → (B, 6, 96) → (B, 96, 6)
```

**PyTorch 基础提示**：
- `nn.Linear(in, out)` 对输入的**最后一维**做线性变换：`output = input @ W.T + b`
- `permute(0, 2, 1)` 交换第 1 和第 2 维：`(B, T, F)` → `(B, F, T)`
- 再 permute 回来得到 `(B, T, F)`，这样 Linear 对每个特征的时间轴做映射

### D-2  Eq.5 —— W(t) 加权混合，预测 $\hat X^0$

$$\boxed{\hat X^0(X^t, t, \theta) = \frac{W(t) \cdot X^t_{1-t:T-t} + (1-b\,W(t))\cdot D}{(1 + c\,W(t))^d}} \tag{5}$$

`Linear.forward` 的完整代码只有几行，但**几乎每一行都和公式对不上**，是除 `q_sample` 外第二个"看代码读不出公式"的地方。逐行拆开：

```python
def forward(self, input_, t, training=True):
    noise = torch.randn_like(input_)
    if not training:
        noise = 0
    input_ += self.w_dev[t[0]] * noise                          # ←(1) 公式 Eq.5 里没有这一项
    x_tmp = self.linear(input_.permute(0,2,1)).permute(0,2,1)   # ←(2) 这才是 Eq.4 的 D
    alpha = self.w[t[0]]                                          # ←(3) 此 alpha = W(t)，不是扩散 √ᾱ！
    output = (alpha*input_ + (1-2*alpha)*x_tmp) / (1-1*alpha)**(1/2)   # ←(4) Eq.5
    return output.to(torch.float32)
```

#### （3）致命的变量名冲突：`alpha` 在这里是 $W(t)$，不是 $\sqrt{\bar\alpha_t}$

这是初学者最容易踩的坑。**同一个名字 `alpha` 在两个文件里含义完全不同**：

| 位置 | `alpha` 指代 | 数值来源 |
|---|---|---|
| `linear.py` 的 `forward` | $W(t)$（可学习权重） | `self.w[t[0]]`，**linear schedule** 的 $\bar\alpha$ |
| `armd.py` 的 `_train_loss` | $\sqrt{\bar\alpha_t}$（扩散系数） | `self.sqrt_alphas_cumprod[t[0]]`，**cosine schedule** |

读 Eq.5 时，公式里的 $W(t)$ 对应代码里的 `alpha`；而 Eq.2/3/6 里的 $\bar\alpha_t$ 是另一套数（见 D-3 的双 schedule 说明）。**两者不要混为一谈。**

#### （4）把硬编码常数代回 Eq.5

代码里 `b=2, c=−1, d=1/2` 被直接写死。代入 Eq.5：

$$\frac{W(t)\cdot X^t + (1-b\,W(t))\cdot D}{(1+c\,W(t))^d}\Bigg|_{b=2,\,c=-1,\,d=1/2} = \frac{W(t)\cdot X^t + (1-2W(t))\cdot D}{(1-W(t))^{1/2}} \;\checkmark$$

与代码 `(alpha*input_ + (1-2*alpha)*x_tmp) / (1-1*alpha)**(1/2)` 逐项一致（`input_`=$X^t$，`x_tmp`=$D$）。

> ⚠️ 因为 $d=1/2$、分母是 $(1-W(t))^{1/2}$，所以 **$W(t)$ 必须 $<1$**，否则开方出 NaN。$W(t)$ 初始化为 $\bar\alpha_t<1$，但它是可学习参数（`w_grad=True`），训练中无约束——这是代码隐含的数值前提，公式里看不出来。

**W(t) 的物理含义**：
- $t$ 大（中间态接近历史）→ $W(t)$ 小 → 更依赖 $D$（距离估计）
- $t$ 小（中间态接近未来）→ $W(t)$ 大 → 更依赖输入 $X^t$ 本身

### D-3  训练时小扰动（Supplemental: Deviation）—— 代码第 (1) 行，Eq.5 里没有

`forward` 第一段在做 Eq.5 之前，先给输入加了一项扰动——**这一项在 Eq.5 中完全不存在**，属于论文补充材料里的训练 trick：

$$X^t_{\text{input}} = X^t + \eta_t \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0,I)$$

```python
noise = torch.randn_like(input_)
if not training:
    noise = 0                          # 推理时关闭扰动 → forward 退化为纯 Eq.5
input_ += self.w_dev[t[0]] * noise     # 扰动系数 = w_dev[t]
```

这里有**三个**只看公式发现不了、必须读代码（甚至要动手验证）才知道的实现细节：

#### （1）扰动系数 `w_dev` 是"逐步 $\alpha=1-\beta$"，不是累积 $\bar\alpha_t$

很容易想当然地以为 $\eta_t=\bar\alpha_t$（和 Eq.13 一样），但代码里：

```python
self.betas_dev  = cosine_beta_schedule(96)      # cosine 的 beta
self.alphas_dev = 1. - self.betas_dev           # 逐步 alpha，注意：没有 cumprod！
self.w_dev      = Parameter(alphas_dev, requires_grad=False)
```

`w_dev[t] = 1 - β_t`（**单步**），而非累积乘积 $\bar\alpha_t=\prod_{k\le t}\alpha_k$。两者数值差异很大：

| $t$ | `w_dev[t] = 1-β_t`（代码实际用） | $\bar\alpha_t$（累积，对比） |
|---|---|---|
| 0  | 0.999 | 0.999 |
| 47 | **0.968** | 0.494 |
| 95 | 0.001 | ≈0 |

所以扰动幅度在大半个 $t$ 区间都接近 1（不小！），只有在 $t\to95$（最接近历史终态）时才骤降到 0。**定性**结论（$t$ 大→扰动小、$t$ 小→扰动大）仍成立，但**定量**上和"$\bar\alpha_t$"完全不同——这是代码与论文符号的一处出入，按代码为准。

#### （2）`input_ += ...` 是原地修改，且会"串改" target（真实的 in-place 别名陷阱）

`+=` 是 in-place 操作，直接改写传入张量的底层存储。而传入的 `input_` 正是 `_train_loss` 里 `x = q_sample(...)` 的**切片视图**，它和回归目标 `target = x_start[:, 96:, :]` **共享同一块 `x_start` 存储且区间重叠**。实测：当 `t_code=0` 时，这一行 `+=` 会改动 `target` 96 个元素中的 95 个。

```python
# _train_loss 内部，两者都是 x_start 的视图：
target = x_start[:, 96:, :]               # 覆盖 x_start[96:192]
x      = x_start[:, 96-index:-index, :]   # 覆盖 x_start[96-index:192-index]，与 target 重叠
model_out = self.output(x, ...)           # 内部 x += w_dev*noise → 同时改了 target 重叠段
```

也就是说，**训练时回归目标本身也被这股扰动"染"了一点**。这是已发布代码的真实行为（不是笔记本的改写），它仍能复现论文指标，但属于"公式上看不出、且容易踩坑"的实现细节。如果你自己改写 `_train_loss`，想避免别名，可在 `q_sample` 里 `return x_middle.clone()` 或在 `forward` 用 `input_ = input_ + ...`（非原地）。推理时 `noise=0`，`input_ += 0` 不改值，等价于跳过，无此问题。

> 下一个代码单元会**实测**这个别名效应，亲眼看到 `target` 被改了多少个元素。

#### （3）三套同名 $\alpha$，务必区分

| 名字/出处 | 含义 | schedule | 是否 cumprod |
|---|---|---|---|
| `linear.py` `self.w` → `alpha` | $W(t)$ 加权权重 | **linear** | 是（`alphas_cumprod`）|
| `linear.py` `self.w_dev` | 扰动系数 $\eta_t$ | **cosine** | **否**（`1-β`）|
| `armd.py` `sqrt_alphas_cumprod` 等 | 扩散 $\sqrt{\bar\alpha_t}$ | **cosine** | 是 |

三处都叫 "alpha/α"，但 schedule 不同、是否累积也不同——这是读 ARMD 源码最大的混淆源，记住这张表即可。


### D-3b  实测：in-place 扰动对 target 的别名影响


In [ ]:
import torch

# 复现 _train_loss 中的视图别名：x 与 target 都是 x_start 的切片视图
pred_len = 96
x_start = torch.arange(192, dtype=torch.float32).reshape(1, 192, 1).clone()

target = x_start[:, pred_len:, :]                 # 真实未来 X^0，覆盖 x_start[96:192]
print("x 是 x_start 的视图吗? ", x_start[:, 0:96, :].data_ptr() == x_start.data_ptr())

for t_code in [0, 47, 95]:
    xs = x_start.clone()                          # 每次重置
    tgt = xs[:, pred_len:, :]
    tgt_before = tgt.clone()
    index = t_code + 1
    x = xs[:, pred_len-index:(-index if index>0 else None), :]   # q_sample 切片（视图）

    # 模拟 Linear.forward 第一行：input_ += w_dev*noise（这里用常数 +1000 放大可见）
    x += 1000.0

    changed = (tgt != tgt_before).sum().item()
    print(f"t_code={t_code:>2} (index={index:>2}): in-place += 改动了 target {changed}/{tgt.numel()} 个元素")

print("\n说明：t_code 越小，x 与 target 的重叠越多，被'串改'的元素越多；")
print("     t_code=95 时 x=完整历史段，与 target 不重叠，target 不受影响。")
print("     真实训练里加的是 w_dev*randn（不是+1000），但别名机制相同。")


### D-4  Eq.6 —— 预测演化趋势 $\hat z$

$$\boxed{\hat z(t,\theta) = \frac{\sqrt{1/\bar\alpha_t}\,X^t_{1-t:T-t} - \hat X^0(X^t,t,\theta)}{\sqrt{1/\bar\alpha_t - 1}}} \tag{6}$$

代码中对应 `predict_noise_from_start`（**函数名沿用 DDPM 的 "noise" 术语，但在 ARMD 语义下是"演化趋势"**）：

```python
def predict_noise_from_start(self, x_t, t, x0):
    return (
        extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - x0
    ) / extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape)
# 等价于: (sqrt(1/abar_t) * x_t - x0) / sqrt(1/abar_t - 1)
# x_t = X^t,  x0 = X_hat^0
```

**一处看代码会困惑的地方：同一个 $\hat z$，代码里有两种写法。** ARMD 在两处算 $\hat z$，公式形态完全不同，却是同一个量：

| 出处 | 代码 | 数学形式 |
|---|---|---|
| `_train_loss`（D-5） | `(x - model_out*alpha) / minus_alpha` | $\dfrac{X^t-\sqrt{\bar\alpha_t}\,\hat X^0}{\sqrt{1-\bar\alpha_t}}$ |
| `predict_noise_from_start`（采样用） | `(sqrt_recip*x_t - x0) / sqrt_recipm1` | $\dfrac{\sqrt{1/\bar\alpha_t}\,X^t-\hat X^0}{\sqrt{1/\bar\alpha_t-1}}$ |

把第二式分子分母同乘 $\sqrt{\bar\alpha_t}$：分子 $\to X^t-\sqrt{\bar\alpha_t}\hat X^0$，分母 $\to\sqrt{\bar\alpha_t}\sqrt{1/\bar\alpha_t-1}=\sqrt{1-\bar\alpha_t}$，**两式逐项相等**。即 `_train_loss` 用的是 Eq.6 的等价改写，`predict_noise_from_start` 用的是 Eq.6 的原式——同一个 $\hat z$。（C-6 后的验证单元会顺带数值确认这一点。）

**`extract` 函数的作用**（初学者必读）：

```python
def extract(a, t, x_shape):
    # a: shape (T,)      - 预计算系数数组，如 sqrt_recip_alphas_cumprod
    # t: shape (B,)      - batch 中每个样本的时间步（相同值）
    # x_shape: (B, 96, 6) - 目标形状，用于广播
    b, *_ = t.shape               # b = batch_size
    out = a.gather(-1, t)         # 按 t 的值从 a 中取对应元素 → shape (B,)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))
    # reshape 为 (B, 1, 1)，方便与 (B, 96, 6) 的张量广播相乘
```

示例：`t = [47, 47, 47]`（B=3，相同时间步），`a[47] = 1.23`
→ `extract(a, t, (3,96,6))` 返回形状 `(3, 1, 1)` 的张量 `[[[1.23]], [[1.23]], [[1.23]]]`。

> **为什么 `_train_loss` 用 `self.X[t[0]]` 标量索引，这里却用 `extract(...)`？** 两者等价：因为 `forward` 里 `t = randint(...).repeat(b)` 让整个 batch 共享同一个 $t$，所以 `self.sqrt_alphas_cumprod[t[0]]` 取一个标量、靠广播作用到整个 batch；`extract` 则显式 gather 成 `(B,1,1)`。同一份系数，两种取法，结果相同。


### D-5  Eq.7 —— 训练目标（L1 Loss）

$$\boxed{\mathcal{L}_\theta = \mathbb{E}_t\!\left[\left|z_t - \hat z(t,\theta)\right|\right]} \tag{7}$$

完整的 `_train_loss` 流程（含形状注释）：

```python
def _train_loss(self, x_start, t, target=None, noise=None, training=True):
    # x_start: (B, 192, 6) — 完整窗口（历史 + 未来）
    noise  = default(noise, lambda: torch.randn_like(x_start))
    target = x_start[:, pred_len:, :]          # (B, 96, 6) — 真实未来 X^0_{1:T}

    x = self.q_sample(x_start=x_start, t=t)   # (B, 96, 6) — 中间态 X^t
    model_out = self.output(x, t, training)    # (B, 96, 6) — 预测 X_hat^0

    # Eq.3 中的系数
    alpha       = self.sqrt_alphas_cumprod[t[0]]           # sqrt(alpha_bar_t), 标量
    minus_alpha = self.sqrt_one_minus_alphas_cumprod[t[0]] # sqrt(1-alpha_bar_t), 标量

    # 真实演化趋势 z_t（Eq.3 变形）
    target_noise = (x - target * alpha) / minus_alpha      # (B, 96, 6)

    # 预测演化趋势 z_hat（Eq.6 变形）
    pred_noise   = (x - model_out * alpha) / minus_alpha   # (B, 96, 6)

    # L1 loss（Eq.7）
    train_loss = self.loss_fn(pred_noise, target_noise, reduction='none')  # (B, 96, 6)
    train_loss = reduce(train_loss, 'b ... -> b (...)', 'mean')            # (B, 96*6)
    train_loss = train_loss * extract(self.loss_weight, t, train_loss.shape)  # 时间步加权
    return train_loss.mean()
```

#### 关键：Eq.7 写成"对 noise 的 L1"，但代码其实是"对 $\hat X^0$ 的加权 L1"

这是又一个"只看公式 Eq.7 看不出、必须读代码才懂"的点。`target_noise` 和 `pred_noise` 共用同一个 $X^t$、同一组系数，只有 $X^0$/$\hat X^0$ 不同。相减时 $X^t$ 整项抵消：

$$z_t - \hat z = \frac{X^t-\sqrt{\bar\alpha_t}X^0}{\sqrt{1-\bar\alpha_t}} - \frac{X^t-\sqrt{\bar\alpha_t}\hat X^0}{\sqrt{1-\bar\alpha_t}} = \frac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}\,(\hat X^0 - X^0)$$

所以 Eq.7 的 $|z_t-\hat z|$ **等价于对 $\hat X^0$ 的回归**，只是带了一个随 $t$ 变化的系数 $\dfrac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}$：

$$\big|z_t-\hat z\big| = \underbrace{\frac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}}_{\text{系数①（相减自带）}}\;\big|\hat X^0 - X^0\big|$$

而代码**之后又乘了一次** `loss_weight`：

$$\mathcal L \;=\; \underbrace{\frac{\sqrt{\alpha_t}\,\sqrt{1-\bar\alpha_t}}{100\,\beta_t}}_{\text{系数②（loss\_weight）}}\;\times\;\underbrace{\frac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}\big|\hat X^0-X^0\big|}_{\text{= }|z_t-\hat z|}$$

两个系数相乘，$\sqrt{1-\bar\alpha_t}$ 约掉，最终对 $|\hat X^0-X^0|$ 的**总有效权重**是 $\dfrac{\sqrt{\alpha_t}\sqrt{\bar\alpha_t}}{100\,\beta_t}$。**这个"双重加权"在 Eq.7 里完全看不出来**——公式只写了 $|z_t-\hat z|$，而代码里 (a) 相减自带系数① + (b) 显式乘 `loss_weight` 系数②。

要点：
- 网络 `model_out` 实际学的是 **$\hat X^0$**（未来初态），不是直接学 noise；"预测 noise"只是 Eq.7 的书写形式。
- `loss_weight = sqrt(alpha_t)*sqrt(1-alpha_bar_t)/beta_t/100` 里的 `/100` 纯是缩放常数（论文未展开），其余因子用来平衡不同 $t$ 的 loss 量级。

> 下一个代码单元用真实张量**数值验证**：直接算的 `l1_loss(pred_noise,target_noise)` 与 `(√ᾱ/√(1-ᾱ))·|X̂⁰-X⁰|` 逐元素相等。


### D-5b  数值验证：Eq.7 的 noise-loss = 加权的 $\hat X^0$-loss


In [ ]:
import torch

torch.manual_seed(0)
B = 4
abar_t = torch.tensor(0.4938)          # 取 t_code=47 的 cosine bar_alpha 作示例
a  = abar_t.sqrt()                     # √ᾱ_t
ma = (1 - abar_t).sqrt()               # √(1-ᾱ_t)

Xt        = torch.randn(B, 96, 6)      # X^t（任意）
X0        = torch.randn(B, 96, 6)      # 真实未来 X^0
X0_hat    = torch.randn(B, 96, 6)      # 网络预测 \hat X^0

# —— 按 _train_loss 的写法：先转成 noise，再 L1 ——
target_noise = (Xt - X0     * a) / ma          # z_t   (Eq.3 变形)
pred_noise   = (Xt - X0_hat * a) / ma          # \hat z (Eq.6 变形)
loss_as_noise = (pred_noise - target_noise).abs()

# —— 解析等价：|z - ẑ| == (√ᾱ/√(1-ᾱ)) * |X̂⁰ - X⁰| ——
loss_as_x0 = (a / ma) * (X0_hat - X0).abs()

print("两种算法逐元素最大差:", (loss_as_noise - loss_as_x0).abs().max().item())
print("→ ≈0 说明：对 noise 的 L1 = (√ᾱ/√(1-ᾱ)) × 对 X̂⁰ 的 L1（系数①）")
print()
print(f"系数① √ᾱ/√(1-ᾱ) (t=47) = {(a/ma).item():.4f}")
print("再乘 loss_weight=√α_t·√(1-ᾱ_t)/(100·β_t)（系数②）后，√(1-ᾱ) 约掉，")
print("对 |X̂⁰-X⁰| 的总有效权重 = √α_t·√ᾱ_t/(100·β_t)。")


---
## Part E：采样/预测过程（Algorithm 2）—— Eq.8–10

> 推理时，从已知**历史序列**出发，通过 devolution 逐步还原**未来预测**。
> 历史序列不是"条件"，而是采样链的**起点**。


### E-1  Eq.8 —— DDIM 风格的完整反向步

类比 DDIM（Song et al., 2021），把 DDPM 的反向步改写为基于 $\hat X^0$ 的形式：

$$X^{t-1}_{2-t:T-t+1} = \sqrt{\bar\alpha_{t-1}}\left[\frac{X^t_{1-t:T-t} - \sqrt{1-\bar\alpha_t}\,\hat z}{\sqrt{\bar\alpha_t}}\right] + \sqrt{1-\bar\alpha_{t-1}-\sigma_t^2}\,\hat z + \sigma_t\varepsilon_t \tag{8}$$

括号内等于 $\hat X^0(X^t, t, \theta)$（由 Eq.5 定义）。

### E-2  Eq.9 —— 确定性简化（去掉随机项）

由于 ARMD 的序列演化是**确定性滑动**，令 $\sigma_t = 0$：

$$\boxed{X^{t-1}_{2-t:T-t+1} = \sqrt{\bar\alpha_{t-1}}\,\hat X^0(X^t,t,\theta) + \sqrt{1-\bar\alpha_{t-1}}\,\hat z(t,\theta)} \tag{9}$$

代码注释：`sigma = 0; noise = 0`（`fast_sample` 中硬编码）。

### E-3  Eq.10 —— 跳步加速采样

$$\boxed{X^{t-k}_{1-t+k:T-t+k} = \sqrt{\bar\alpha_{t-k}}\,\hat X^0(X^t,t,\theta) + \sqrt{1-\bar\alpha_{t-k}}\,\hat z(t,\theta)} \tag{10}$$

Eq.10 里的 $k$ 是一次反向更新跨过的时间间隔；源码没有直接把 `sampling_timesteps` 当成这个 $k$。
源码语义是：`sampling_timesteps` 表示**反向更新次数**，再用
`linspace(-1, T-1, steps=sampling_timesteps+1)` 生成实际时间网格。

例如 `T=96, sampling_timesteps=2` 时：

```python
times = [95, 47, -1]
time_pairs = [(95, 47), (47, -1)]
```

也就是执行 2 次反向网络调用；第一步从 95 跳到 47，最后一步遇到 `-1` 哨兵后直接输出
$\hat X^0$。论文/补充材料会在候选集合 $\{1,2,3,4,6,8,12\}$ 中选择这个**采样步数**。
本教程使用 `sampling_timesteps=2`。

`fast_sample` 的实现（含详细注释）：

```python
@torch.no_grad()
def fast_sample(self, x, clip_denoised=True):
    # x: (B, 192, 6) — 完整测试窗口（含历史+未来，但只用历史半段）
    batch = x.shape[0]
    # 生成 DDIM 跳步网格：sampling_timesteps 控制反向更新次数
    times = torch.linspace(-1, self.num_timesteps - 1, steps=self.sampling_timesteps + 1)
    times = list(reversed(times.int().tolist()))
    # 相邻步对：[(T-1, T-2/T-k), ..., (k, 0), (0, -1)]
    time_pairs = list(zip(times[:-1], times[1:]))

    img = x[:, :pred_len, :]   # (B, 96, 6) — 历史段，Algorithm 2 的起点 X^T

    for time, time_next in time_pairs:
        time_cond = torch.full((batch,), time, device=device, dtype=torch.long)
        pred_noise, x_start, *_ = self.model_predictions(img, time_cond, clip_x_start=clip_denoised)
        # x_start : (B, 96, 6) — \hat{X}^0
        # pred_noise: (B, 96, 6) — \hat{z}(t, theta)

        if time_next < 0:
            img = x_start   # 最后一步，直接输出 X_hat^0
            continue

        alpha_next = self.alphas_cumprod[time_next]   # bar_alpha_{t-k}
        sigma = 0    # 确定性：sigma_t = 0
        noise = 0    # 不加随机噪声
        c = (1 - alpha_next - sigma ** 2).sqrt()      # = sqrt(1 - bar_alpha_{t-k})
        img = x_start * alpha_next.sqrt() + c * pred_noise   # Eq.(10)

    return img   # (B, 96, 6) — 最终预测 X_hat^0_{1:T}
```

**Algorithm 2 伪代码对照**：

| 论文步骤 | 代码 |
|---|---|
| 输入：历史序列 $X^T_{-T+1:0}$ | `img = x[:, :96, :]` |
| 输入：trained $R(\cdot)$, 采样步数, $\bar\alpha_{0:T}$ | `model`, `sampling_timesteps`, `alphas_cumprod` |
| for $t = T$ to $0$ by $\Delta t$ | `for time, time_next in time_pairs:` |
| 用 $R(\cdot)$ 得到 $\hat X^0$, $\hat z$ | `x_start, pred_noise = model_predictions(img, t)` |
| Eq.(10) 更新 | `img = x_start * alpha_next.sqrt() + c * pred_noise` |
| 输出 $X^0_{1:T}$ | `return img` |

### E-4  `fast_sample` 里几处"公式上看不出"的实现细节

和 `q_sample` 一样，`fast_sample` 有几行代码无法和 Eq.8–10 直接对上，逐一说明：

**(1) 更新式 `img = x_start*alpha_next.sqrt() + c*pred_noise` 怎么就是 Eq.10？**

代码变量到公式的映射（注意 `sigma=0` 被硬编码）：

| 代码 | 公式（Eq.10，$\sigma=0$） |
|---|---|
| `x_start` | $\hat X^0(X^t,t,\theta)$（`model_predictions` 返回） |
| `pred_noise` | $\hat z(t,\theta)$（即 `predict_noise_from_start` 的输出） |
| `alpha_next.sqrt()` | $\sqrt{\bar\alpha_{t-k}}$ |
| `c = (1-alpha_next-sigma**2).sqrt()` | $\sqrt{1-\bar\alpha_{t-k}-\sigma_t^2}\xrightarrow{\sigma=0}\sqrt{1-\bar\alpha_{t-k}}$ |

代回：`x_start*√ᾱ_next + √(1-ᾱ_next)*pred_noise` $=\sqrt{\bar\alpha_{t-k}}\hat X^0+\sqrt{1-\bar\alpha_{t-k}}\hat z$ ＝ Eq.10。✓ 注意 Eq.8 方括号内"$\frac{X^t-\sqrt{1-\bar\alpha_t}\hat z}{\sqrt{\bar\alpha_t}}$"在代码里**不再出现**——因为它恒等于 $\hat X^0$，而 `model_predictions` 已直接给出 $\hat X^0$（`x_start`），无需再算方括号。这正是 DDIM 把反向步重写成"$\hat X^0$ + $\hat z$ 线性组合"的好处。

**(2) `pred_noise`/`alpha_next` 的时间索引：这里直接用 `time`/`time_next`，没有 `q_sample` 的 +1**

`q_sample` 切片用 `index=t_code+1`，但这里 `self.alphas_cumprod[time_next]`、`model_predictions(img, time)` 都是**直接拿 `time` 当下标**，没有 +1。看似矛盾，其实一致：`q_sample` 的 `+1` 是把"论文步 $t$"换算成"滑动几步"（滑 $t$ 步要从拼接序列偏移 $t$）；而系数数组 `alphas_cumprod` 是 0-based，`alphas_cumprod[time]` 恰好就是论文第 $time{+}1$ 步的 $\bar\alpha$。两处最终都指向同一个论文步，只是一个数"滑动步数"、一个数"数组下标"。

**(3) `times = linspace(-1, T-1, steps=sampling_timesteps+1)` 里的 `-1` 是什么？**

```python
times = torch.linspace(-1, self.num_timesteps - 1, steps=self.sampling_timesteps + 1)
times = list(reversed(times.int().tolist()))     # 例: k=2 → [95, 47, -1]
time_pairs = list(zip(times[:-1], times[1:]))     # → [(95,47), (47,-1)]
```

末尾的 `-1` 是个**哨兵值**：当 `time_next < 0`（最后一对），说明已经走到序列最前端，此时不再做 Eq.10 的线性组合，而是 `img = x_start` 直接输出 $\hat X^0$（见代码 `if time_next < 0`）。这一步在 Algorithm 2 里对应"循环结束、输出 $X^0_{1:T}$"，公式 Eq.10 本身没有体现这个边界处理。

**(4) 起点是历史，不是高斯噪声**

`img = x[:, :pred_len, :]`（注意源码里 `img = torch.randn(...)` 那行被注释掉了）。这是 ARMD 区别于普通扩散模型的本质：采样链起点 $X^T$ ＝**已知的历史序列**，而非 $\mathcal N(0,I)$，所以只需极少步（本教程 `sampling_timesteps=2`）即可。

**(5) `clip_denoised=True` 在 `fast_sample` 路径里实际没有裁剪**

`fast_sample` 把 `clip_denoised` 传给 `model_predictions(..., clip_x_start=clip_denoised)`，但源码里真正裁剪的那行被注释掉了：

```python
maybe_clip = partial(torch.clamp, min=-2, max=2) if clip_x_start else identity
x_start = self.output(x, t, training)
# x_start = maybe_clip(x_start)   # 注释状态：没有执行
```

因此 Stock 复现走 `fast_sample` 时，`x_start` 不会被 clamp 到 `[-2, 2]`。这和 `p_mean_variance`
传统路径里的 `x_start.clamp_(-1., 1.)` 不同；Stock paper-style 配置使用的是 `fast_sample`。

**(6) `eta` 参数也被硬覆盖**

源码先按 DDIM 公式计算：

```python
sigma = eta * ((1 - alpha / alpha_next) * (1 - alpha_next) / (1 - alpha)).sqrt()
```

但下一行立刻 `sigma = 0`，再把 `noise = 0`。所以即使外部传入非零 `eta`，当前
`fast_sample` 仍是确定性的；Eq.8 的随机项不会进入 Stock 评估。


---
## Part F：代码实现（Beta Schedules / model_utils / Linear / ARMD）

> 以下 Cell 内嵌仓库核心代码。为了让 notebook 可以作为单文件运行，少量位置做了机械改写；
> 所有改写都列在下表，避免把 standalone 版本误认为逐字源码。

| Notebook 章节 | 原始文件 | standalone 中的处理 | 行为是否应与 Stock 复现一致 |
|---|---|---|---|
| A-3 `CustomDataset` | `Utils/Data_utils/real_datasets.py` | 手写 Stock-only replica；去掉随机缺失 mask/fMRI 泛化/.npy 依赖路径 | 是，覆盖 Stock 预测所需行为 |
| F-2 `model_utils` | `Models/autoregressive_diffusion/model_utils.py` | 只提取 `exists/default/identity/extract` 四个 ARMD 实际使用函数 | 是 |
| F-3 `Linear` | `Models/autoregressive_diffusion/linear.py` | 去掉未使用的仓库相对 import 和 einops import | 是 |
| F-4 `ARMD` | `Models/autoregressive_diffusion/armd.py` | 去掉仓库相对 import；依赖前面 cell 已定义的 `Linear/default/identity/extract` | 是 |
| G-1 LR scheduler | `engine/lr_sch.py` | 只保留 `ReduceLROnPlateauWithWarmup`，省略当前 Stock 配置不用的另一个 scheduler | 是 |
| G-2 `Trainer` | `engine/solver.py` | 把 `instantiate_from_config` 调度器构造改为直接构造；去掉日志参数统计 import | 是 |
| G-3 DataLoader | `Data/build_dataloader.py` | 用已创建的 `train_ds/test_ds` 直接构造 DataLoader，不再从 YAML instantiate | 是 |
| I-2 `main.py` | `main.py` | 原文只展示不执行；上文 H/I 已把等价流程展开 | 作为参考 |


### F-1  Beta Schedules（`linear.py` 头部）

已在 B-4 定义，此处已可用。


### F-2  `model_utils` 辅助函数

来自 `Models/autoregressive_diffusion/model_utils.py`，仅提取 ARMD 实际使用的 4 个函数。

| 函数 | 签名 | 说明 |
|------|------|------|
| `exists` | `exists(x)` | 判断 `x is not None` |
| `default` | `default(val, d)` | 若 `val` 为 None 则返回默认值 `d` |
| `identity` | `identity(t, ...)` | 恒等函数（占位用） |
| `extract` | `extract(a, t, x_shape)` | 按批次时间步索引系数并 reshape 用于广播 |


In [ ]:
def exists(x):
    return x is not None


def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d


def identity(t, *args, **kwargs):
    return t


def extract(a, t, x_shape):
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))


### F-3  `Linear` 类（standalone 嵌入版）—— Eq.4–5 的 Devolution 网络

来自 `Models/autoregressive_diffusion/linear.py`，已去掉仓库相对 import 和 einops（不再使用）。

**`__init__` 中的两个 schedule**：

```
self.w    = Parameter(alphas_cumprod from linear_beta_schedule)  # 可学习，初始化为线性 schedule 的 bar_alpha
self.w_dev = Parameter(alphas_dev from cosine_beta_schedule)      # 不可学习，用于训练扰动
```

为什么 `w` 和 `w_dev` 用不同 schedule？
- `w` 作为 Eq.5 的 $W(t)$，初始化来自 `linear_beta_schedule(96)` 的累积乘积，并且 `requires_grad=True`；
- `w_dev` 不是 $\eta_{0:t}$ 的累积量，而是 cosine schedule 的单步 `1-beta_t`；它在大半个时间步区间接近 1，
  只有接近末端时快速降到 0。这个行为在 D-3 已详细拆解。

**`forward` 中的形状流**：

```
input_: (B, 96, 6)
  → + w_dev[t]*noise                   # 训练扰动（推理时 noise=0）
  → permute(0,2,1) → (B, 6, 96)
  → nn.Linear(96,96) → (B, 6, 96)     # Eq.4: D = Linear(X^t)
  → permute(0,2,1) → x_tmp: (B, 96, 6)
  → (w[t]*input_ + (1-2*w[t])*x_tmp) / (1-w[t])^0.5   # Eq.5
output: (B, 96, 6)                      # X_hat^0
```


In [ ]:
import math
import torch
import numpy as np
import torch.nn.functional as F

from torch import nn
def linear_beta_schedule(timesteps):
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float64)

def cosine_beta_schedule(timesteps, s=0.008):
    """
    cosine schedule
    as proposed in https://openreview.net/forum?id=-NEXDKk8gZ
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float64)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

timesteps = 96

class Linear(nn.Module):
    def __init__(
        self,
        n_feat,
        n_channel,
        w_grad=True,
        **kwargs
    ):
        super().__init__()
        self.linear = nn.Linear(n_channel, n_channel)
        self.betas = linear_beta_schedule(96)
        self.betas_dev = cosine_beta_schedule(96)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_dev = 1. - self.betas_dev
        self.w = torch.nn.Parameter(torch.FloatTensor(self.alphas_cumprod.numpy()), requires_grad=w_grad)
        self.w_dev = torch.nn.Parameter(torch.FloatTensor(self.alphas_dev.numpy()), requires_grad=False)

    def forward(self, input_, t, training=True):
        noise = torch.randn_like(input_)
        if not training:
            noise=0
        input_+= self.w_dev[t[0]]*noise
        x_tmp = self.linear(input_.permute(0,2,1)).permute(0,2,1)
        alpha = self.w[t[0]]
        output = (alpha*input_ + (1-2*alpha)*x_tmp) / (1-1*alpha)**(1/2)
        #if not training:
            #print('alpha:',alpha)
            #print('para:',1-1*alpha)
            #print('dis:',x_tmp.mean())
            #print('loss:',((1-1*alpha)*x_tmp).mean())

        output = output.to(torch.float32)

        return output


### F-4  `ARMD` 类（standalone 嵌入版）—— 主逻辑

来自 `Models/autoregressive_diffusion/armd.py`，已去掉仓库相对 import。

**`__init__` 中预注册的 buffers**（`register_buffer` 说明）：

`register_buffer(name, tensor)` 把张量注册为 buffer：
- **不是可训练参数**（不在 `model.parameters()` 中）
- 但跟模型一起移动（`model.to(device)` 时自动迁移）
- `model.state_dict()` 中有它（可保存/加载）

预计算的系数（对应论文 Eq.13 的各种变形）：

| Buffer 名称 | 公式 | 用途 |
|---|---|---|
| `betas` | $\beta_t$ | 噪声方差 |
| `alphas_cumprod` | $\bar\alpha_t$ | 主 schedule |
| `sqrt_alphas_cumprod` | $\sqrt{\bar\alpha_t}$ | `_train_loss` 中乘 target |
| `sqrt_one_minus_alphas_cumprod` | $\sqrt{1-\bar\alpha_t}$ | `_train_loss` 分母 |
| `sqrt_recip_alphas_cumprod` | $\sqrt{1/\bar\alpha_t}$ | `predict_noise_from_start` |
| `sqrt_recipm1_alphas_cumprod` | $\sqrt{1/\bar\alpha_t-1}$ | `predict_noise_from_start` 分母 |
| `loss_weight` | $\frac{\sqrt{\alpha_t}\sqrt{1-\bar\alpha_t}}{100\beta_t}$ | loss 时间步加权 |

**关键细节**：`t = torch.randint(0, T, (1,)).repeat(b)` —— 整个 batch 共享同一个时间步 t，而不是每个样本独立采样。所以下游大量代码使用 `t[0]` 标量索引；如果改成每样本独立 `t`，`q_sample`、`Linear.forward` 和 `_train_loss` 都需要同步改写。


In [ ]:
import math
import torch
import torch.nn.functional as F

from torch import nn
from einops import reduce
from tqdm.auto import tqdm
from functools import partial


# gaussian diffusion trainer class

pred_len = 96

def linear_beta_schedule(timesteps):
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float64)


def cosine_beta_schedule(timesteps, s=0.008):
    """
    cosine schedule
    as proposed in https://openreview.net/forum?id=-NEXDKk8gZ
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float64)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)


class ARMD(nn.Module):
    def __init__(
            self,
            seq_length,
            feature_size,
            n_layer_enc=3,
            n_layer_dec=6,
            d_model=None,
            timesteps=1000,
            sampling_timesteps=None,
            loss_type='l1',
            beta_schedule='cosine',
            n_heads=4,
            mlp_hidden_times=4,
            eta=0.,
            attn_pd=0.,
            resid_pd=0.,
            w_grad=True,
            use_revin=False,
            **kwargs
    ):
        super(ARMD, self).__init__()

        self.eta = eta
        self.seq_length = seq_length
        self.feature_size = feature_size
        # RevIN: normalize each window by its OWN lookback (context) statistics so a
        # trending series becomes per-window stationary. Stats are re-applied to the
        # prediction so metrics stay in the outer-scaler space. Off by default.
        self.use_revin = use_revin

        self.model = Linear(n_feat=feature_size, n_channel=seq_length, w_grad=w_grad, **kwargs)

        if beta_schedule == 'linear':
            betas = linear_beta_schedule(timesteps)
        elif beta_schedule == 'cosine':
            betas = cosine_beta_schedule(timesteps)
        else:
            raise ValueError(f'unknown beta schedule {beta_schedule}')

        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.)

        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)
        self.loss_type = loss_type

        # sampling related parameters

        self.sampling_timesteps = default(
            sampling_timesteps, timesteps)  # default num sampling timesteps to number of timesteps at training

        assert self.sampling_timesteps <= timesteps
        self.fast_sampling = self.sampling_timesteps < timesteps

        # helper function to register buffer from float64 to float32

        register_buffer = lambda name, val: self.register_buffer(name, val.to(torch.float32))

        register_buffer('betas', betas)
        register_buffer('alphas_cumprod', alphas_cumprod)
        register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)

        # calculations for diffusion q(x_t | x_{t-1}) and others

        register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1. - alphas_cumprod))
        register_buffer('log_one_minus_alphas_cumprod', torch.log(1. - alphas_cumprod))
        register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1. / alphas_cumprod))
        register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1. / alphas_cumprod - 1))

        # calculations for posterior q(x_{t-1} | x_t, x_0)

        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)

        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)

        register_buffer('posterior_variance', posterior_variance)

        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain

        register_buffer('posterior_log_variance_clipped', torch.log(posterior_variance.clamp(min=1e-20)))
        register_buffer('posterior_mean_coef1', betas * torch.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))
        register_buffer('posterior_mean_coef2', (1. - alphas_cumprod_prev) * torch.sqrt(alphas) / (1. - alphas_cumprod))

        # calculate reweighting
        
        register_buffer('loss_weight', torch.sqrt(alphas) * torch.sqrt(1. - alphas_cumprod) / betas / 100)

    def predict_noise_from_start(self, x_t, t, x0):
        return (
                (extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - x0) /
                extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape)
        )
    
    def predict_start_from_noise(self, x_t, t, noise):
        return (
            extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
            extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise
        )

    def q_posterior(self, x_start, x_t, t):
        posterior_mean = (
                extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +
                extract(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        posterior_variance = extract(self.posterior_variance, t, x_t.shape)
        posterior_log_variance_clipped = extract(self.posterior_log_variance_clipped, t, x_t.shape)
        return posterior_mean, posterior_variance, posterior_log_variance_clipped
    
    def output(self, x, t, training=False):
        model_output = self.model(x, t, training=training)
        return model_output

    def model_predictions(self, x, t, clip_x_start=False, training=False):
        if training:
            training = False       #padding masks = 1
        maybe_clip = partial(torch.clamp, min=-2, max=2) if clip_x_start else identity
        x_start = self.output(x, t, training)
        #x_start = maybe_clip(x_start)
        pred_noise = self.predict_noise_from_start(x, t, x_start)
        return pred_noise, x_start

    def p_mean_variance(self, x, t, clip_denoised=True):
        _, x_start = self.model_predictions(x, t)
        if clip_denoised:
            x_start.clamp_(-1., 1.)
        model_mean, posterior_variance, posterior_log_variance = \
            self.q_posterior(x_start=x_start, x_t=x, t=t)
        return model_mean, posterior_variance, posterior_log_variance, x_start

    def p_sample(self, x, t: int, clip_denoised=True):
        batched_times = torch.full((x.shape[0],), t, device=x.device, dtype=torch.long)
        model_mean, _, model_log_variance, x_start = \
            self.p_mean_variance(x=x, t=batched_times, clip_denoised=clip_denoised)
        noise = torch.randn_like(x) if t > 0 else 0.  # no noise if t == 0
        pred_img = model_mean + (0.5 * model_log_variance).exp() * noise
        return pred_img, x_start

    @torch.no_grad()
    def sample(self, x):
        device = self.betas.device
        shape = x.shape
        img = x[:,:pred_len,:]
        #img = torch.randn(shape, device=device)

        for t in tqdm(reversed(range(0, self.num_timesteps)),
                      desc='sampling loop time step', total=self.num_timesteps):
            img, _ = self.p_sample(img, t)
        return img

    @torch.no_grad()
    def fast_sample(self, x, clip_denoised=True):
        shape = x.shape
        batch, device, total_timesteps, sampling_timesteps, eta = \
            shape[0], self.betas.device, self.num_timesteps, self.sampling_timesteps, self.eta

        # [-1, 0, 1, 2, ..., T-1] when sampling_timesteps == total_timesteps
        times = torch.linspace(-1, total_timesteps - 1, steps=sampling_timesteps + 1)

        times = list(reversed(times.int().tolist()))
        time_pairs = list(zip(times[:-1], times[1:]))  # [(T-1, T-2), (T-2, T-3), ..., (1, 0), (0, -1)]
        #img = torch.randn(shape, device=device)
        img = x[:,:pred_len,:]

        for time, time_next in tqdm(time_pairs, desc='sampling loop time step'):
            time_cond = torch.full((batch,), time, device=device, dtype=torch.long)
            pred_noise, x_start, *_ = self.model_predictions(img, time_cond, clip_x_start=clip_denoised)
            if time_next < 0:
                img = x_start
                continue
            alpha = self.alphas_cumprod[time]
            alpha_next = self.alphas_cumprod[time_next]
            sigma = eta * ((1 - alpha / alpha_next) * (1 - alpha_next) / (1 - alpha)).sqrt()
            sigma = 0
            c = (1 - alpha_next - sigma ** 2).sqrt()
            noise = 0
            img = x_start * alpha_next.sqrt() + \
                  c * pred_noise + \
                  sigma * noise

        return img

    def _revin_stats(self, x, eps=1e-5):
        """Per-window mean/std over the lookback (context) portion x[:, :pred_len, :]."""
        ctx = x[:, :pred_len, :]
        mu = ctx.mean(dim=1, keepdim=True)
        sigma = torch.sqrt(ctx.var(dim=1, keepdim=True, unbiased=False) + eps)
        return mu, sigma

    def generate_mts(self, x):
        sample_fn = self.fast_sample if self.fast_sampling else self.sample
        if self.use_revin:
            mu, sigma = self._revin_stats(x)
            pred = sample_fn((x - mu) / sigma)   # predict in per-window normalized space
            return pred * sigma + mu             # de-normalize back to outer-scaler space
        return sample_fn(x)

    @property
    def loss_fn(self):
        if self.loss_type == 'l1':
            return F.l1_loss
        elif self.loss_type == 'l2':
            return F.mse_loss
        else:
            raise ValueError(f'invalid loss type {self.loss_type}')

    def q_sample(self, x_start, t, noise=None):
        index = int(t[0])+1
        x_middle = x_start[:,pred_len-index:-index,:]
        return x_middle

    def _train_loss(self, x_start, t, target=None, noise=None, training=True):
        noise = default(noise, lambda: torch.randn_like(x_start))
        if target is None:
            target = x_start[:,pred_len:,:]
        target = x_start[:,pred_len:,:]
        x = self.q_sample(x_start=x_start, t=t, noise=noise)  # noise sample
        model_out = self.output(x, t, training)
        alpha = self.sqrt_alphas_cumprod[t[0]]
        minus_alpha = self.sqrt_one_minus_alphas_cumprod[t[0]]
        target_noise = (x - target*alpha)/minus_alpha
        pred_noise = (x - model_out*alpha)/minus_alpha

        train_loss = self.loss_fn(pred_noise, target_noise, reduction='none')

        train_loss = reduce(train_loss, 'b ... -> b (...)', 'mean')
        train_loss = train_loss * extract(self.loss_weight, t, train_loss.shape)
        return train_loss.mean()

    def forward(self, x, **kwargs):
        b, c, n, device, feature_size, = *x.shape, x.device, self.feature_size
        assert n == feature_size, f'number of variable must be {feature_size}'
        if self.use_revin:
            # Train in the same per-window normalized space the sampler predicts in.
            mu, sigma = self._revin_stats(x)
            x = (x - mu) / sigma
        t = torch.randint(0, self.num_timesteps, (1,), device=device).repeat(b).long()
        return self._train_loss(x_start=x, t=t, **kwargs)

    def langevin_fn(
        self,
        coef,
        partial_mask,
        tgt_embs,
        learning_rate,
        sample,
        mean,
        sigma,
        t,
        coef_=0.
    ):
    
        if t[0].item() < self.num_timesteps * 0.05:
            K = 0
        elif t[0].item() > self.num_timesteps * 0.9:
            K = 3
        elif t[0].item() > self.num_timesteps * 0.75:
            K = 2
            learning_rate = learning_rate * 0.5
        else:
            K = 1
            learning_rate = learning_rate * 0.25

        input_embs_param = torch.nn.Parameter(sample)

        with torch.enable_grad():
            for i in range(K):
                optimizer = torch.optim.Adagrad([input_embs_param], lr=learning_rate)
                optimizer.zero_grad()

                x_start = self.output(x=input_embs_param, t=t)

                if sigma.mean() == 0:
                    logp_term = coef * ((mean - input_embs_param) ** 2 / 1.).mean(dim=0).sum()
                    infill_loss = (x_start[partial_mask] - tgt_embs[partial_mask]) ** 2
                    infill_loss = infill_loss.mean(dim=0).sum()
                else:
                    logp_term = coef * ((mean - input_embs_param)**2 / sigma).mean(dim=0).sum()
                    infill_loss = (x_start[partial_mask] - tgt_embs[partial_mask]) ** 2
                    infill_loss = (infill_loss/sigma.mean()).mean(dim=0).sum()
            
                loss = logp_term + infill_loss
                loss.backward()
                optimizer.step()
                epsilon = torch.randn_like(input_embs_param.data)
                input_embs_param = torch.nn.Parameter((input_embs_param.data + coef_ * sigma.mean().item() * epsilon).detach())

        sample[~partial_mask] = input_embs_param.data[~partial_mask]
        return sample
    

if __name__ == '__main__':
    pass


---
## Part G：训练支持代码（LR 调度 / Trainer / DataLoader）


### G-1  `ReduceLROnPlateauWithWarmup`（`engine/lr_sch.py`）

学习率调度策略（论文实验使用此调度器）：

1. **Warmup 阶段**（前 `warmup=500` 步）：lr 从初始值线性升到 `warmup_lr=8e-4`
2. **Plateau 监控阶段**（之后）：若 loss 在 `patience=4000` 步内无改善，lr 乘以 `factor=0.5`，最小不低于 `min_lr=1e-5`

**PyTorch 提示**：标准 PyTorch 调度器需要调用 `scheduler.step()`，此自定义类也一样，但参数是当前 loss 值（而非 epoch 数）。


In [ ]:
import math
from torch import inf
from torch.optim.optimizer import Optimizer


class ReduceLROnPlateauWithWarmup(object):
    """Reduce learning rate when a metric has stopped improving.
    Models often benefit from reducing the learning rate by a factor
    of 2-10 once learning stagnates. This scheduler reads a metrics
    quantity and if no improvement is seen for a 'patience' number
    of epochs, the learning rate is reduced.

    Args:
        optimizer (Optimizer): Wrapped optimizer.
        mode (str): One of `min`, `max`. In `min` mode, lr will
            be reduced when the quantity monitored has stopped
            decreasing; in `max` mode it will be reduced when the
            quantity monitored has stopped increasing. Default: 'min'.
        factor (float): Factor by which the learning rate will be
            reduced. new_lr = lr * factor. Default: 0.1.
        patience (int): Number of epochs with no improvement after
            which learning rate will be reduced. For example, if
            `patience = 2`, then we will ignore the first 2 epochs
            with no improvement, and will only decrease the LR after the
            3rd epoch if the loss still hasn't improved then.
            Default: 10.
        threshold (float): Threshold for measuring the new optimum,
            to only focus on significant changes. Default: 1e-4.
        threshold_mode (str): One of `rel`, `abs`. In `rel` mode,
            dynamic_threshold = best * ( 1 + threshold ) in 'max'
            mode or best * ( 1 - threshold ) in `min` mode.
            In `abs` mode, dynamic_threshold = best + threshold in
            `max` mode or best - threshold in `min` mode. Default: 'rel'.
        cooldown (int): Number of epochs to wait before resuming
            normal operation after lr has been reduced. Default: 0.
        min_lr (float or list): A scalar or a list of scalars. A
            lower bound on the learning rate of all param groups
            or each group respectively. Default: 0.
        eps (float): Minimal decay applied to lr. If the difference
            between new and old lr is smaller than eps, the update is
            ignored. Default: 1e-8.
        verbose (bool): If ``True``, prints a message to stdout for
            each update. Default: ``False``.
        warmup_lr: float or None, the learning rate to be touched after warmup
        warmup: int, the number of steps to warmup
    """

    def __init__(self, optimizer, mode='min', factor=0.1, patience=10,
                 threshold=1e-4, threshold_mode='rel', cooldown=0,
                 min_lr=0, eps=1e-8, verbose=False, warmup_lr=None,
                 warmup=0):

        if factor >= 1.0:
            raise ValueError('Factor should be < 1.0.')
        self.factor = factor

        # Attach optimizer
        if not isinstance(optimizer, Optimizer):
            raise TypeError('{} is not an Optimizer'.format(
                type(optimizer).__name__))
        self.optimizer = optimizer

        if isinstance(min_lr, list) or isinstance(min_lr, tuple):
            if len(min_lr) != len(optimizer.param_groups):
                raise ValueError("expected {} min_lrs, got {}".format(
                    len(optimizer.param_groups), len(min_lr)))
            self.min_lrs = list(min_lr)
        else:
            self.min_lrs = [min_lr] * len(optimizer.param_groups)

        self.patience = patience
        self.verbose = verbose
        self.cooldown = cooldown
        self.cooldown_counter = 0
        self.mode = mode
        self.threshold = threshold
        self.threshold_mode = threshold_mode

        self.warmup_lr = warmup_lr
        self.warmup = warmup
        
        self.best = None
        self.num_bad_epochs = None
        self.mode_worse = None  # the worse value for the chosen mode
        self.eps = eps
        self.last_epoch = 0
        self._init_is_better(mode=mode, threshold=threshold,
                             threshold_mode=threshold_mode)
        self._reset()

    def _prepare_for_warmup(self):
        if self.warmup_lr is not None:
            if isinstance(self.warmup_lr, (list, tuple)):
                if len(self.warmup_lr) != len(self.optimizer.param_groups):
                    raise ValueError("expected {} warmup_lrs, got {}".format(
                        len(self.optimizer.param_groups), len(self.warmup_lr)))
                self.warmup_lrs = list(self.warmup_lr)
            else:
                self.warmup_lrs = [self.warmup_lr] * len(self.optimizer.param_groups)
        else:
            self.warmup_lrs = None
        if self.warmup > self.last_epoch:
            curr_lrs = [group['lr'] for group in self.optimizer.param_groups]
            self.warmup_lr_steps = [max(0, (self.warmup_lrs[i] - curr_lrs[i])/float(self.warmup)) for i in range(len(curr_lrs))]
        else:
            self.warmup_lr_steps = None

    def _reset(self):
        """Resets num_bad_epochs counter and cooldown counter."""
        self.best = self.mode_worse
        self.cooldown_counter = 0
        self.num_bad_epochs = 0

    def step(self, metrics):
        # convert `metrics` to float, in case it's a zero-dim Tensor
        current = float(metrics)
        epoch = self.last_epoch + 1
        self.last_epoch = epoch

        if epoch <= self.warmup:
            self._increase_lr(epoch)
        else:
            if self.is_better(current, self.best):
                self.best = current
                self.num_bad_epochs = 0
            else:
                self.num_bad_epochs += 1

            if self.in_cooldown:
                self.cooldown_counter -= 1
                self.num_bad_epochs = 0  # ignore any bad epochs in cooldown

            if self.num_bad_epochs > self.patience:
                self._reduce_lr(epoch)
                self.cooldown_counter = self.cooldown
                self.num_bad_epochs = 0

            self._last_lr = [group['lr'] for group in self.optimizer.param_groups]

    def _reduce_lr(self, epoch):
        for i, param_group in enumerate(self.optimizer.param_groups):
            old_lr = float(param_group['lr'])
            new_lr = max(old_lr * self.factor, self.min_lrs[i])
            if old_lr - new_lr > self.eps:
                param_group['lr'] = new_lr
                if self.verbose:
                    print('Epoch {:5d}: reducing learning rate'
                          ' of group {} to {:.4e}.'.format(epoch, i, new_lr))

    def _increase_lr(self, epoch):
        # used for warmup
        for i, param_group in enumerate(self.optimizer.param_groups):
            old_lr = float(param_group['lr'])
            new_lr = max(old_lr + self.warmup_lr_steps[i], self.min_lrs[i])
            param_group['lr'] = new_lr
            if self.verbose:
                print('Epoch {:5d}: increasing learning rate'
                        ' of group {} to {:.4e}.'.format(epoch, i, new_lr))

    @property
    def in_cooldown(self):
        return self.cooldown_counter > 0

    def is_better(self, a, best):
        if self.mode == 'min' and self.threshold_mode == 'rel':
            rel_epsilon = 1. - self.threshold
            return a < best * rel_epsilon

        elif self.mode == 'min' and self.threshold_mode == 'abs':
            return a < best - self.threshold

        elif self.mode == 'max' and self.threshold_mode == 'rel':
            rel_epsilon = self.threshold + 1.
            return a > best * rel_epsilon

        else:  # mode == 'max' and epsilon_mode == 'abs':
            return a > best + self.threshold

    def _init_is_better(self, mode, threshold, threshold_mode):
        if mode not in {'min', 'max'}:
            raise ValueError('mode ' + mode + ' is unknown!')
        if threshold_mode not in {'rel', 'abs'}:
            raise ValueError('threshold mode ' + threshold_mode + ' is unknown!')

        if mode == 'min':
            self.mode_worse = inf
        else:  # mode == 'max':
            self.mode_worse = -inf

        self.mode = mode
        self.threshold = threshold
        self.threshold_mode = threshold_mode

        self._prepare_for_warmup()

    def state_dict(self):
        return {key: value for key, value in self.__dict__.items() if key != 'optimizer'}

    def load_state_dict(self, state_dict):
        self.__dict__.update(state_dict)
        self._init_is_better(mode=self.mode, threshold=self.threshold, threshold_mode=self.threshold_mode)


### G-2  `Trainer` 类（`engine/solver.py`，Algorithm 1 的完整实现）

**修改说明**（相比原始仓库）：
- `instantiate_from_config(cfg['scheduler'])` → 直接构造 `ReduceLROnPlateauWithWarmup(**params)`（避免依赖 YAML 解析）
- 删除 `sys.path.append` 和 `from Utils.io_utils import ...`

**关键实现细节**：

| 组件 | 说明 |
|------|------|
| `Adam([...], betas=[0.9, 0.96])` | 标准 Adam，$\beta_1=0.9, \beta_2=0.96$ |
| `EMA(model, decay=0.995)` | 指数移动平均：推理时用 EMA 权重而非原始权重，更稳定 |
| `clip_grad_norm_(model.parameters(), 1.0)` | 梯度裁剪：防止梯度爆炸 |
| `gradient_accumulate_every=2` | 梯度累积：等效于 batch_size × 2，但内存占用不变 |
| `cycle(dataloader)` | 将 DataLoader 变成无限迭代器（训练 steps 而非 epochs） |
| `results_folder = config_folder + f"_{seq_len}"` | 检查点目录自动拼接 seq_length |

**EMA 说明**：
```
EMA 权重 = 0.995 * 上一步EMA权重 + 0.005 * 当前模型权重
```
推理时 `trainer.sample_forecast` 调用 `self.ema.ema_model.generate_mts(x)`（而非 `self.model`）。


In [ ]:
import os
import sys
import time
import torch
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from ema_pytorch import EMA
from torch.optim import Adam
from torch.nn.utils import clip_grad_norm_



def cycle(dl):
    while True:
        for data in dl:
            yield data


class Trainer(object):
    def __init__(self, config, args, model, dataloader, logger=None):
        super().__init__()
        self.model = model
        self.device = self.model.betas.device
        self.train_num_steps = config['solver']['max_epochs']
        self.gradient_accumulate_every = config['solver']['gradient_accumulate_every']
        self.save_cycle = config['solver']['save_cycle']
        self.dl = cycle(dataloader['dataloader'])
        self.step = 0
        self.milestone = 0
        self.args = args
        self.logger = logger

        self.results_folder = Path(config['solver']['results_folder'] + f'_{model.seq_length}')
        os.makedirs(self.results_folder, exist_ok=True)

        start_lr = config['solver'].get('base_lr', 1.0e-4)
        ema_decay = config['solver']['ema']['decay']
        ema_update_every = config['solver']['ema']['update_interval']

        self.opt = Adam(filter(lambda p: p.requires_grad, self.model.parameters()), lr=start_lr, betas=[0.9, 0.96])
        self.ema = EMA(self.model, beta=ema_decay, update_every=ema_update_every).to(self.device)

        p = dict(config['solver']['scheduler']['params'])
        p['optimizer'] = self.opt
        self.sch = ReduceLROnPlateauWithWarmup(**p)

        # parameter info logging omitted in standalone
        self.log_frequency = 100

    def save(self, milestone, verbose=False):
        if self.logger is not None and verbose:
            self.logger.log_info('Save current model to {}'.format(str(self.results_folder / f'checkpoint-{milestone}.pt')))
        data = {
            'step': self.step,
            'model': self.model.state_dict(),
            'ema': self.ema.state_dict(),
            'opt': self.opt.state_dict(),
        }
        torch.save(data, str(self.results_folder / f'checkpoint-{milestone}.pt'))

    def load(self, milestone, verbose=False):
        if self.logger is not None and verbose:
            self.logger.log_info('Resume from {}'.format(str(self.results_folder / f'checkpoint-{milestone}.pt')))
        device = self.device
        data = torch.load(str(self.results_folder / f'checkpoint-{milestone}.pt'), map_location=device)
        self.model.load_state_dict(data['model'])
        self.step = data['step']
        self.opt.load_state_dict(data['opt'])
        self.ema.load_state_dict(data['ema'])
        self.milestone = milestone

    def train(self):
        device = self.device
        step = 0
        if self.logger is not None:
            tic = time.time()
            self.logger.log_info('{}: start training...'.format(self.args.name), check_primary=False)

        with tqdm(initial=step, total=self.train_num_steps) as pbar:
            while step < self.train_num_steps:
                total_loss = 0.
                for _ in range(self.gradient_accumulate_every):
                    data = next(self.dl).to(device)
                    loss = self.model(data, target=data)
                    loss = loss / self.gradient_accumulate_every
                    loss.backward()
                    total_loss += loss.item()

                pbar.set_description(f'loss: {total_loss:.6f}')

                clip_grad_norm_(self.model.parameters(), 1.0)
                self.opt.step()
                self.sch.step(total_loss)
                self.opt.zero_grad()
                self.step += 1
                step += 1
                self.ema.update()

                with torch.no_grad():
                    if self.step != 0 and self.step % self.save_cycle == 0:
                        self.milestone += 1
                        self.save(self.milestone)
                        # self.logger.log_info('saved in {}'.format(str(self.results_folder / f'checkpoint-{self.milestone}.pt')))
                    
                    if self.logger is not None and self.step % self.log_frequency == 0:
                        # info = '{}: train'.format(self.args.name)
                        # info = info + ': Epoch {}/{}'.format(self.step, self.train_num_steps)
                        # info += ' ||'
                        # info += '' if loss_f == 'none' else ' Fourier Loss: {:.4f}'.format(loss_f.item())
                        # info += '' if loss_r == 'none' else ' Reglarization: {:.4f}'.format(loss_r.item())
                        # info += ' | Total Loss: {:.6f}'.format(total_loss)
                        # self.logger.log_info(info)
                        self.logger.add_scalar(tag='train/loss', scalar_value=total_loss, global_step=self.step)

                pbar.update(1)

        print('training complete')
        if self.logger is not None:
            self.logger.log_info('Training done, time: {:.2f}'.format(time.time() - tic))

    def sample(self, num, size_every, shape=None):
        if self.logger is not None:
            tic = time.time()
            self.logger.log_info('Begin to sample...')
        samples = np.empty([0, shape[0], shape[1]])
        #print(samples.shape)
        num_cycle = int(num // size_every) + 1

        for _ in range(num_cycle):
            sample = self.ema.ema_model.generate_mts(batch_size=size_every)
            #print(sample.shape)
            samples = np.row_stack([samples, sample.detach().cpu().numpy()])
            torch.cuda.empty_cache()

        if self.logger is not None:
            self.logger.log_info('Sampling done, time: {:.2f}'.format(time.time() - tic))
        return samples

    def sample_forecast(self, raw_dataloader, shape=None):
        if self.logger is not None:
            tic = time.time()
            self.logger.log_info('Begin to sample...')
        samples = np.empty([0, shape[0], shape[1]])
        reals = np.empty([0, shape[0], shape[1]])
        #print(samples.shape)

        for idx, batch in enumerate(raw_dataloader):
            if len(batch)==2:
                x, t_m = batch
                x, t_m = x.to(self.device), t_m.to(self.device)
            else:
                x = batch
                x = x.to(self.device)
            sample = self.ema.ema_model.generate_mts(x)
            #print(sample.shape)
            samples = np.row_stack([samples, sample.detach().cpu().numpy()])
            #reals = None
            reals = np.row_stack([reals, x[:,shape[0]:,:].detach().cpu().numpy()])
            torch.cuda.empty_cache()

        if self.logger is not None:
            self.logger.log_info('Sampling done, time: {:.2f}'.format(time.time() - tic))
        return samples, reals


### G-3  `build_dataloader`（`Data/build_dataloader.py`）

把 `CustomDataset` 包装成 PyTorch `DataLoader`。
原始 `Data/build_dataloader.py` 从 YAML config 里调用 `instantiate_from_config(...)` 创建 dataset；
standalone 里 `train_ds/test_ds` 已经在 A-4 创建好，所以这里直接把 dataset 对象传给 DataLoader。
DataLoader 的关键行为（batch、shuffle、drop_last、测试 mask）保持一致。

**训练 vs 测试的差异**：

| 参数 | 训练 (`build_dataloader`) | 测试 (`build_dataloader_cond`) |
|------|------|------|
| `batch_size` | `config['dataloader']['batch_size']` = 128 | `sample_size` = 256 |
| `shuffle` | `True` | `False` |
| `drop_last` | `True`（丢弃不完整的末尾 batch） | `False` |
| 模式 | `period='train'` | `period='test'`, `predict_length=96` |


In [ ]:
def build_dataloader(dataset, batch_size, shuffle=True):
    """训练 DataLoader（对应 Data/build_dataloader.py::build_dataloader）。"""
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=shuffle,   # 训练时丢弃不完整 batch，测试时保留
    )

def build_dataloader_cond(test_dataset, sample_size=256):
    """测试 DataLoader（对应 Data/build_dataloader.py::build_dataloader_cond）。"""
    return torch.utils.data.DataLoader(
        test_dataset,
        batch_size=sample_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

# 创建实际用的 DataLoader
train_loader = build_dataloader(train_ds, batch_size=128, shuffle=True)
test_loader  = build_dataloader_cond(test_ds, sample_size=256)

print(f"训练 DataLoader: {len(train_loader)} 个 batch / epoch  (每 batch 128 窗口)")
print(f"测试 DataLoader: {len(test_loader)} 个 batch          (每 batch 256 窗口)")
print(f"训练 batch 示例形状: {next(iter(train_loader)).shape}")
x_test, mask = next(iter(test_loader))
print(f"测试 batch 示例 x 形状: {x_test.shape}  mask 形状: {mask.shape}")
print(f"  mask[:, :96, :] == True (历史可见)")
print(f"  mask[:, 96:, :] == False (未来区域标记为预测目标)")
print("注意：sample_forecast 不读取 mask；评估真值直接来自 x[:, 96:, :]")


---
## Part H：训练（Algorithm 1）与评估（Algorithm 2）

> **选择运行模式**（在下一个 Cell 中设置 `QUICK_TEST`）：
>
> | 模式 | steps | 预计时间（GPU） | 指标 |
> |------|-------|----------------|------|
> | `True`（快速验证） | 100 | ~3 分钟 | 不可与论文对比 |
> | `False`（完整复现） | 2000 | ~20 分钟 | 对齐论文 Table 1 |


### H-1  超参数配置（对应 `Config/stock_paper.yaml`）

| 超参数 | 值 | 说明 |
|---|---|---|
| `seq_length` | 96 | 历史 = 预测步数 |
| `timesteps` | 96 | 最大扩散步数 $T$ |
| `sampling_timesteps` | 2 | 反向更新次数（从 {1,2,3,4,6,8,12} 验证集选取；实际跳步网格由 `linspace` 生成） |
| `loss_type` | `l1` | 对应 Eq.7（绝对值范数） |
| `beta_schedule` | `cosine` | ARMD buffers 初始化 |
| `use_revin` | `True` | 每个窗口用历史半段做 RevIN，缓解 Stock 趋势漂移 |
| `base_lr` | 1e-3 | Adam 初始学习率 |
| `max_epochs` | 2000 | 总 optimizer steps |
| `gradient_accumulate_every` | 2 | 梯度累积步数 |
| `batch_size` | 128 | 有效 batch = 128 × 2 = 256 |
| `ema.decay` | 0.995 | EMA 衰减系数 |
| `warmup` | 500 | LR warmup 步数 |

`use_revin=True` 是当前仓库 `Config/stock_paper.yaml` 的关键设置：模型先用每个样本历史半段
`x[:, :96, :]` 的均值/标准差把完整窗口归一化，在这个窗口内归一化空间训练和采样；
推理输出再乘回同一组标准差并加回均值。这样评估仍在外层 `StandardScaler` 的 z-score 空间，
但模型不必直接拟合跨年份价格水平漂移。


In [ ]:
import os, random
import numpy as np
import torch
from torch.nn.utils import clip_grad_norm_

# ══════════════════════════════════════════════════════════════════════════
# 选择训练模式
QUICK_TEST = False   # True=100步/3min验证流程；False=2000步/复现论文
# ══════════════════════════════════════════════════════════════════════════

MAX_EPOCHS = 100 if QUICK_TEST else 2000
SAMPLING_TIMESTEPS = 2     # sampling_steps from {1,2,3,4,6,8,12}
LOSS_TYPE = "l1"           # Eq.7 是 L1 loss
USE_REVIN = True            # mirrors Config/stock_paper.yaml

if QUICK_TEST:
    print(f"[快速验证模式] {MAX_EPOCHS} steps，指标不可与论文对比")
else:
    print(f"[完整复现模式] {MAX_EPOCHS} steps，对应 Config/stock_paper.yaml")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(2023)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"设备: {DEVICE}")

# ── 创建 ARMD 模型（对应 main.py 的 instantiate_from_config(configs["model"])）
model = ARMD(
    seq_length=SEQ_LEN,          # 96
    feature_size=N_FEAT,         # 6（Stock）
    timesteps=96,                # T = 96
    sampling_timesteps=SAMPLING_TIMESTEPS,
    loss_type=LOSS_TYPE,
    beta_schedule="cosine",
    w_grad=True,                 # W(t) 可学习
    use_revin=USE_REVIN,          # per-window lookback normalization
).to(DEVICE)
model.fast_sampling = True       # 启用 fast_sample (DDIM 跳步)

n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数: {n_params:,}  "
      f"(Linear.linear: {SEQ_LEN*SEQ_LEN + SEQ_LEN:,}; "
      f"Linear.w: {SEQ_LEN}; Linear.w_dev: {SEQ_LEN})")
print(f"RevIN: {model.use_revin}  (stats from x[:, :{SEQ_LEN}, :], output de-normalized before scoring)")

# ── solver 配置（镜像 Config/stock_paper.yaml）
config = {
    "solver": {
        "max_epochs": MAX_EPOCHS,
        "gradient_accumulate_every": 2,
        "save_cycle": 10**9,            # 教程不保存检查点
        "results_folder": str(REPO_ROOT / "Checkpoints_standalone_nb"),
        "base_lr": 1e-3,
        "ema": {"decay": 0.995, "update_interval": 10},
        "scheduler": {
            "params": {
                "mode": "min", "factor": 0.5, "patience": 4000,
                "min_lr": 1e-5, "threshold": 0.1, "threshold_mode": "rel",
                "warmup_lr": 8e-4, "warmup": 500, "verbose": False,
            }
        },
    }
}

class Args:
    name = "armd_tutorial"
    save_dir = str(REPO_ROOT / "forecasting_exp_standalone")

args = Args()
os.makedirs(args.save_dir, exist_ok=True)

def cycle_loader(dl):
    """无限迭代器（对应 engine/solver.py 的 cycle 函数）。"""
    while True:
        for x in dl:
            yield x

trainer = Trainer(
    config=config, args=args, model=model,
    dataloader={"dataloader": cycle_loader(train_loader)}, logger=None,
)
print("Trainer 创建完成。")


### H-2  训练循环（手动展开 `Trainer.train()`，含 Algorithm 1 对照）

手动展开而非调用 `trainer.train()`，以便实时记录 loss 历史；关键训练口径与项目训练循环保持一致。

**Algorithm 1 对照**（每个 optimizer step 的逻辑）：

```
Algorithm 1 (单步):
  1. x_start ← DataLoader (完整窗口 (B, 192, 6))      → next(trainer.dl)
  2. if use_revin: normalize by lookback stats       → ARMD.forward
  3. t ← Uniform({1,...,T})                          → `randint(..., (1,)).repeat(b)`，整 batch 共享同一个 t
  4. X^t ← q_sample(x_start, t)                      → ARMD.q_sample
  5. X_hat^0 ← Linear(X^t, t)                        → ARMD.output
  6. z_hat ← (X^t - sqrt(abar_t)*X_hat0)/sqrt(1-abar_t) → ARMD._train_loss 中 Eq.6 等价写法
  7. L = L1(z_t, z_hat)                              → ARMD._train_loss
  8. update theta ← Adam.step()                      → trainer.opt.step()
  9. update EMA                                      → trainer.ema.update()
```

第 2 步不是论文 Algorithm 1 的显式行，而是当前仓库为 Stock 配置加入的实现层预处理。
它不会改变指标空间：`generate_mts` 会把预测反变换回外层 z-score 空间后再交给 `sample_forecast` 计算 MSE/MAE。

第 3 步也有一个源码细节：论文写的是采样一个 $t$，仓库实现为
`t = torch.randint(0, self.num_timesteps, (1,), device=device).repeat(b).long()`。
因此一个训练 batch 内所有样本共享同一个时间步；后续 `_train_loss` 用 `t[0]` 取系数，正是依赖这个实现。


In [ ]:
from tqdm.auto import tqdm

loss_history: list[float] = []

pbar = tqdm(range(MAX_EPOCHS), desc="train", smoothing=0.05)
for step in range(MAX_EPOCHS):
    total_loss = 0.0
    # ── 梯度累积（gradient_accumulate_every=2）──────────────────────────
    # 等效于 batch_size*2 的有效 batch，但实际只前向一次 128 个样本
    for _ in range(trainer.gradient_accumulate_every):
        data = next(trainer.dl).to(trainer.device)   # (128, 192, 6)
        # ARMD.forward: 随机采样 t → q_sample → Linear → _train_loss
        loss = trainer.model(data, target=data)       # 标量 loss
        loss = loss / trainer.gradient_accumulate_every   # 归一化
        loss.backward()                               # 累积梯度
        total_loss += loss.item()

    clip_grad_norm_(trainer.model.parameters(), 1.0) # 梯度裁剪（防爆炸）
    trainer.opt.step()                                # Adam 更新参数
    trainer.sch.step(total_loss)                      # LR 调度监控 loss
    trainer.opt.zero_grad()                           # 清零梯度
    trainer.step += 1
    trainer.ema.update()                              # EMA 权重更新

    loss_history.append(total_loss)
    if step % max(1, MAX_EPOCHS//20) == 0:
        pbar.set_description(f"loss: {total_loss:.6f}")
    pbar.update(1)

pbar.close()
print(f"训练完成: {len(loss_history)} steps  最终 loss: {loss_history[-1]:.6f}")


### H-3  训练 Loss 曲线


In [ ]:
import matplotlib.pyplot as plt, numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(loss_history, lw=0.6, color="#2196F3")
axes[0].set_title(f"Loss 曲线（{len(loss_history)} steps）")
axes[0].set_xlabel("optimizer step"); axes[0].set_ylabel("L1 loss"); axes[0].grid(alpha=0.3)

if len(loss_history) >= 50:
    w = 50
    smooth = np.convolve(loss_history, np.ones(w)/w, mode="valid")
    axes[1].plot(smooth, lw=1.0, color="#E91E63")
    axes[1].axvline(min(500, len(smooth)), color="grey", ls="--", lw=1, label="warmup end")
    axes[1].set_title(f"平滑 Loss（{w}步移动平均）")
    axes[1].set_xlabel("optimizer step"); axes[1].legend(); axes[1].grid(alpha=0.3)
else:
    axes[1].plot(loss_history, lw=1.0); axes[1].set_title("Loss（步数不足50）"); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"初始 loss: {loss_history[0]:.6f}  最终 loss: {loss_history[-1]:.6f}")


### H-4  评估（Algorithm 2，10 次采样平均）

**评估协议**（对应 `main.py`，论文 Table 1 使用相同方式）：
1. 对测试集重复 10 次采样
2. 每次用不同随机种子（`2023+run`）
3. 平均 10 次的 MSE / MAE

由于 `fast_sample` 中 `sigma=0; noise=0`（确定性），10 次结果理论上完全一致；保留 10 次只是与 `main.py` 协议对齐。

**指标说明**：MSE / MAE 在 **z-score 归一化空间**计算，**不反变换**到原始价格尺度。

**mask 说明**：`build_dataloader_cond` 会让测试集返回 `(x, mask)`，其中 `mask[:, 96:, :] = False`
表示未来半段是预测目标。但 `Trainer.sample_forecast` 在预测任务中并不使用这个 mask：

```python
if len(batch) == 2:
    x, t_m = batch
    x, t_m = x.to(device), t_m.to(device)
sample = self.ema.ema_model.generate_mts(x)
reals = np.row_stack([reals, x[:, shape[0]:, :].detach().cpu().numpy()])
```

也就是说，mask 只是沿用了 infill/predict 共用数据接口；Stock 预测评估的真实值直接取
`x[:, 96:, :]`，预测输入则在 `generate_mts/fast_sample` 内只使用 `x[:, :96, :]` 作为历史起点。


In [ ]:
import random, numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

shape = [SEQ_LEN, N_FEAT]  # [96, 6]
mse_runs, mae_runs = [], []
samples_last = reals_last = None

for run in range(10):
    set_seed(2023 + run)
    # sample_forecast 内部: fast_sample 用历史 x[:, :96, :] → 预测 (B, 96, 6)
    # reals: 真实未来段 x[:, 96:, :] 从 test_loader 收集
    samples, reals = trainer.sample_forecast(test_loader, shape=shape)
    mse_runs.append(mean_squared_error(samples.reshape(-1), reals.reshape(-1)))
    mae_runs.append(mean_absolute_error(samples.reshape(-1), reals.reshape(-1)))
    samples_last, reals_last = samples, reals

mse = float(np.mean(mse_runs))
mae = float(np.mean(mae_runs))
print("=" * 58)
print(f"ARMD on Stock — {SAMPLING_TIMESTEPS}-step DDIM, 10次采样平均")
print(f"  MSE = {mse:.4f}    MAE = {mae:.4f}")
print(f"  per-run MSE: {[round(m,4) for m in mse_runs]}")
print("=" * 58)
print()
print("论文 Table 1 参考值 (Stock, z-score): MSE=0.235  MAE=0.269")
if QUICK_TEST:
    print("[提示] 快速验证模式，指标偏高属正常；改 QUICK_TEST=False 复现论文")


### H-4b  论文 Table 1、仓库配置与本 notebook 的比较口径

对 Stock 结果做比较时，先确认“比较的到底是哪一套口径”：

| 项目 | 论文 Table 1 / Supplemental | 当前仓库 `Config/stock_paper.yaml` | 本 standalone 默认设置 |
|---|---|---|---|
| 数据 | Stock，96→96，多变量 6 特征 | `Data/datasets/stock_data.csv`，3685×6 | 同左；若 CSV 缺失会生成占位随机游走，不能比论文 |
| 切分 | 70/10/20 时间顺序切分 | `three_split=True, train_ratio=0.7, val_ratio=0.1` | 同左 |
| 归一化 | z-score 空间计算 MSE/MAE | `StandardScaler.fit(全部行)`；`norm_on_train=False`；`use_revin=True` 做窗口内训练/推理 | 同左 |
| Loss | L1 | `loss_type='l1'` | 同左 |
| 训练步数 | 2000 iterations | `max_epochs=2000` | `QUICK_TEST=False` 时同左；`True` 仅 smoke test |
| batch | 128，梯度累积 2 | `batch_size=128`, `gradient_accumulate_every=2` | 同左 |
| 采样步数 | 从 {1..12} 在验证集选择 | 本仓库 sweep 记录 `sampling_timesteps=2` | 同左 |
| 评估 | 10 次采样平均 | `main.py` seeds 2023..2032；但 `fast_sample` 确定性，10 次应相同 | 同左 |
| 论文参考值 | Stock ARMD: MSE 0.235, MAE 0.269 | 当前配置没有随文件附带已执行结果；需重新训练评估 | notebook 运行值取决于本机训练和 `QUICK_TEST` |

为什么本地 paper-style run 的 MSE 可能不同于论文 Table 1？

- `results/stock_paper_comparison.json` 中的 MSE≈0.0937/MAE≈0.2481 是历史本地记录，来自旧的
  `Config/stock_paper_full_data_eval.yaml` / `Config/stock_paper_ts2.yaml` 口径；这些旧配置没有显式
  `use_revin=True`，不能直接当作当前 `Config/stock_paper.yaml` 的结果。
- Stock CSV 构建或窗口边界可能与作者内部实验有细微差异。
- 论文表格可能包含多训练 seed 或不同 checkpoint 选择；本仓库 `main.py` 固定单个训练 seed。
- `fast_sample` 本身是确定性的，所谓“10 次采样平均”在当前实现中主要是协议对齐。
- `QUICK_TEST=True` 只验证流程，不是复现实验；只有 `QUICK_TEST=False` 才进入 2000-step paper-style 口径。

因此：本 notebook 的核心目标是**把论文公式、仓库实现、复现实验口径打通**；若要严肃报告数值，
请同时记录 config、CSV 来源、训练 seed、checkpoint/EMA、`sampling_timesteps` 和 `QUICK_TEST` 状态。


### H-5  预测可视化


In [ ]:
import matplotlib.pyplot as plt, numpy as np

n_show = 4; feat_show = min(2, N_FEAT)
idx_list = np.random.default_rng(42).choice(samples_last.shape[0], size=n_show, replace=False)

fig, axes = plt.subplots(n_show, feat_show, figsize=(6*feat_show, 2.8*n_show), sharex=True)
if n_show == 1: axes = np.array([axes])
if feat_show == 1: axes = axes[:, None]

hist_segs = test_ds.samples[:, :SEQ_LEN, :]  # 历史段

for r, idx in enumerate(idx_list):
    for c in range(feat_show):
        ax = axes[r, c]
        xh = np.arange(SEQ_LEN); xf = np.arange(SEQ_LEN, 2*SEQ_LEN)
        ax.plot(xh, hist_segs[idx,:,c],    color="#555",    lw=0.9,
                label="历史" if (r==0 and c==0) else None)
        ax.plot(xf, reals_last[idx,:,c],   color="#1f77b4", lw=1.0,
                label="真实未来" if (r==0 and c==0) else None)
        ax.plot(xf, samples_last[idx,:,c], color="#d62728", lw=1.3, ls="--",
                label="ARMD预测" if (r==0 and c==0) else None)
        ax.axvline(SEQ_LEN-0.5, color="grey", ls="--", lw=0.8)
        ax.grid(alpha=0.3); ax.set_title(f"窗口#{int(idx)} feat{c}", fontsize=9)
        if c == 0: ax.set_ylabel("z-score", fontsize=8)

axes[0,0].legend(loc="upper left", fontsize=8)
fig.suptitle(f"ARMD 预测 (z-score空间)  MSE={mse:.4f} MAE={mae:.4f}", y=1.01)
plt.tight_layout(); plt.show()


---
## Part I：`main.py` 等价代码与消融实验分析


### I-1  `main.py` 完整流程（参考）

`main.py` 是项目的实际入口，与本教程的等价关系：

```python
# main.py 核心逻辑（伪代码，与本教程各步骤对应）

# 1. 加载配置（YAML → dict）→ 本教程: config dict 直接内联
configs = load_yaml_config(args.config_path)

# 2. 创建模型（instantiate_from_config）→ 本教程: ARMD(...)
model = instantiate_from_config(configs['model']).to(device)
model.fast_sampling = True

# 3. 创建训练 DataLoader → 本教程: build_dataloader(train_ds, ...)
dataloader_info = build_dataloader(configs, args)

# 4. 创建 Trainer → 本教程: Trainer(config, args, model, dataloader)
trainer = Trainer(config=configs, args=args, model=model,
                  dataloader={'dataloader': dataloader})

# 5. 训练（Algorithm 1）→ 本教程: 手动展开训练循环
trainer.train()

# 6. 创建测试 DataLoader → 本教程: build_dataloader_cond(test_ds, ...)
test_dataloader_info = build_dataloader_cond(configs, args)

# 7. 评估（Algorithm 2，10次平均）→ 本教程: for run in range(10): sample_forecast
mse_runs, mae_runs = [], []
for run in range(10):
    set_seed(2023 + run)
    sample, real_ = trainer.sample_forecast(test_dataloader, shape=[seq_len, feat_num])
    mse_runs.append(mean_squared_error(...))
    mae_runs.append(mean_absolute_error(...))
mse, mae = np.mean(mse_runs), np.mean(mae_runs)
print(mse, mae)
```


### I-2  `main.py` 原始代码（参考）


In [ ]:
# 以下是 main.py 的原始代码，用于参考。
# 在本教程中我们已将其完全展开并内嵌。
# 此 Cell 仅展示，不执行（因为仓库 import 在 standalone 中不可用）
'''
import os
import torch
import numpy as np
import random
import argparse

import warnings
warnings.filterwarnings("ignore")

from engine.solver import Trainer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from Utils.io_utils import load_yaml_config, instantiate_from_config
from Data.build_dataloader import build_dataloader, build_dataloader_cond

def set_seed(seed):
    """
    Set the random seed for reproducibility.
    
    Parameters:
    - seed (int): The seed value.
    """
    # Set the seed for Python's built-in random module
    random.seed(seed)
    
    # Set the seed for NumPy
    np.random.seed(seed)
    
    # Set the seed for PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional steps for CuDNN backend
    os.environ['PYTHONHASHSEED'] = str(seed)

# Example usage:
set_seed(2023)

#class Args_Example:
#    def __init__(self) -> None:
#        self.config_path = './Config/etth.yaml'
#        self.save_dir = './forecasting_exp'
#        self.gpu = 0
#        os.makedirs(self.save_dir, exist_ok=True)

class Args_Example:
    def __init__(self, config_path, save_dir, gpu):
        self.config_path = config_path
        self.save_dir = save_dir
        self.gpu = gpu
        os.makedirs(self.save_dir, exist_ok=True)

def parse_arguments():
    parser = argparse.ArgumentParser(description="Process configuration and directories.")
    parser.add_argument('--config_path', type=str, required=True,
                        help='Path to the configuration file.')
    parser.add_argument('--save_dir', type=str, default='./forecasting_exp',
                        help='Directory to save experiment results.')
    parser.add_argument('--gpu', type=int, default=0,
                        help='Specify which GPU to use.')
    
    args = parser.parse_args()
    return args

if __name__ == "__main__":
    #args =  Args_Example()
    args_parsed = parse_arguments()
    args = Args_Example(args_parsed.config_path, args_parsed.save_dir, args_parsed.gpu)
    seq_len = 96
    configs = load_yaml_config(args.config_path)
    device = torch.device(f'cuda:{args.gpu}' if torch.cuda.is_available() else 'cpu')
    model = instantiate_from_config(configs['model']).to(device)
    #model.use_ff = False
    model.fast_sampling = True
    #configs['solver']['max_epochs']=100
    dataloader_info = build_dataloader(configs, args)
    dataloader = dataloader_info['dataloader']
    trainer = Trainer(config=configs, args=args, model=model, dataloader={'dataloader':dataloader})
    trainer.train()
    args.mode = 'predict'
    args.pred_len = seq_len
    test_dataloader_info = build_dataloader_cond(configs, args)
    test_scaled = test_dataloader_info['dataset'].samples
    scaler = test_dataloader_info['dataset'].scaler
    seq_length, feat_num = seq_len*2, test_scaled.shape[-1]
    pred_length = seq_len
    real = test_scaled
    test_dataset = test_dataloader_info['dataset']
    test_dataloader = test_dataloader_info['dataloader']
    # Paper: metrics averaged over 10 sampling runs (deterministic fast_sample -> identical runs).
    mse_runs, mae_runs = [], []
    for run in range(10):
        torch.manual_seed(2023 + run)
        np.random.seed(2023 + run)
        random.seed(2023 + run)
        sample, real_ = trainer.sample_forecast(test_dataloader, shape=[seq_len, feat_num])
        mse_runs.append(mean_squared_error(sample.reshape(-1), real_.reshape(-1)))
        mae_runs.append(mean_absolute_error(sample.reshape(-1), real_.reshape(-1)))
    mse, mae = float(np.mean(mse_runs)), float(np.mean(mae_runs))
    print(mse, mae)
    cfg_l = args.config_path.replace("\\", "/").lower()
    if "stock" in cfg_l:
        print(
            "Paper reference (Table 1, ARMD on Stock, z-score): MSE=0.235 MAE=0.269 "
            "(https://arxiv.org/abs/2412.09328)"
        )
'''
print('[参考代码已显示，不执行]')


### I-3  消融实验分析（对应论文 Table 4）

论文对 ARMD 进行了 5 个消融实验（在 7 个数据集上，共 14 个设置）：

| 消融变体 | 最优次数 | 原因分析 |
|---|---|---|
| **ARMD（完整模型）** | **11/14** | 基准 |
| 插值方法（Interpolation） | 0/14 | 线性插值 $X^t = X^0 + (X^T-X^0)t/T$ 破坏了时间序列的**自然演化规律**，中间态不再代表真实过渡状态 |
| T-embedding 方法 | 0/14 | 把时间步 $t$ 作为条件注入（传统 DDPM 做法），网络无法利用**滑动带来的结构信息** |
| Transformer 骨干网络 | 3/14 | 参数量更大但未必更好；Linear 足够捕捉时间序列的**线性相关性**，且更高效 |
| 去除小扰动（Deviation） | 0/14 | 训练时无扰动 → **过拟合**到固定中间态，泛化能力下降 |
| 添加随机噪声（sampling）| 0/14 | 推理时加噪声 → 破坏了前向过程的**确定性**，导致预测不稳定 |

**关键设计选择总结**：
1. **滑动 > 插值**：真实的时间演化是平滑滑动，不是两端线性混合。
2. **Linear > Transformer**：对于时间序列的短程线性映射，简单线性层已足够且快 10× 以上。
3. **有扰动 > 无扰动**：少量随机性提升训练多样性，类似 Dropout 的正则效果。
4. **确定性采样 > 随机采样**：时间序列演化本身是确定性的，不需要随机噪声。


### I-4  线性模型实验：同一数据窗口上的 OLS / Ridge 对照

为了判断 ARMD 的收益是否来自复杂的扩散式反向过程，先做一个更直接的线性基线：

- 输入：同一个 Stock 滑窗的历史半段 `x[:, :96, :]`
- 目标：同一个滑窗的未来半段 `x[:, 96:, :]`
- 特征：把 `[96, 6]` 展平成 `576` 维
- 模型：多输出普通最小二乘（OLS）与 Ridge 回归
- 指标：与上文 ARMD 一样，在 z-score 归一化空间中把所有元素展平后计算 MSE / MAE

这里还加入一个 Persistence 基线：重复最后一个历史观测值作为整个未来预测。它对接近随机游走的序列很强，因此是必要的 sanity check。

公平性口径：如果上文 ARMD 使用 `use_revin=True`，线性模型也先按每个窗口历史半段做同样的
RevIN，再把预测反变换回外层 z-score 空间打分；否则 OLS/Ridge 会和 ARMD 处在不同的输入空间。

注意：这一节是**同一仓库、同一窗口、同一指标空间**下的 sanity check；
它不是论文 Table 1 的外部模型排行榜项。若 `QUICK_TEST=True`，ARMD 行只代表快速流程验证，不代表正式性能。


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error


def split_context_future(samples: np.ndarray, seq_len: int):
    """[N, 2*seq_len, F] -> flattened context, flattened future, future tensor."""
    context = samples[:, :seq_len, :]
    future = samples[:, seq_len:, :]
    n = samples.shape[0]
    return context.reshape(n, -1), future.reshape(n, -1), future


def evaluate_flat(pred, true):
    return (
        float(mean_squared_error(pred.reshape(-1), true.reshape(-1))),
        float(mean_absolute_error(pred.reshape(-1), true.reshape(-1))),
    )


def revin_stats_np(samples: np.ndarray, seq_len: int, eps: float = 1e-5):
    """Per-window lookback mean/std, matching ARMD._revin_stats."""
    ctx = samples[:, :seq_len, :]
    mu = ctx.mean(axis=1, keepdims=True)
    sigma = np.sqrt(ctx.var(axis=1, keepdims=True) + eps)
    return mu, sigma


def fit_predict_linear(regressor, train_samples, test_samples, seq_len: int, use_revin: bool):
    """Fit context->future; with RevIN, train/predict in per-window normalized space."""
    if use_revin:
        mu_train, sig_train = revin_stats_np(train_samples, seq_len)
        mu_test, sig_test = revin_stats_np(test_samples, seq_len)
        train_work = (train_samples - mu_train) / sig_train
        test_work = (test_samples - mu_test) / sig_test
        X_train_i, y_train_i, _ = split_context_future(train_work, seq_len)
        X_test_i, _, _ = split_context_future(test_work, seq_len)
        regressor.fit(X_train_i, y_train_i)
        pred_norm = regressor.predict(X_test_i).reshape(test_samples.shape[0], seq_len, test_samples.shape[-1])
        return pred_norm * sig_test + mu_test

    X_train_i, y_train_i, _ = split_context_future(train_samples, seq_len)
    X_test_i, _, _ = split_context_future(test_samples, seq_len)
    regressor.fit(X_train_i, y_train_i)
    return regressor.predict(X_test_i).reshape(test_samples.shape[0], seq_len, test_samples.shape[-1])


train_samples = np.asarray(train_ds.samples)
test_samples = np.asarray(test_ds.samples)
X_train, y_train, _ = split_context_future(train_samples, SEQ_LEN)
X_test, y_test, y_test_seq = split_context_future(test_samples, SEQ_LEN)
use_revin_for_linear = bool(globals().get("USE_REVIN", getattr(model, "use_revin", False)))

linear_results = []

# Persistence: repeat the last observed value for all future steps.
last_value = test_samples[:, SEQ_LEN - 1:SEQ_LEN, :]
persistence_pred = np.repeat(last_value, SEQ_LEN, axis=1)
linear_results.append(("Persistence (last value)", *evaluate_flat(persistence_pred, y_test_seq)))

ols = LinearRegression()
ols_pred_seq = fit_predict_linear(ols, train_samples, test_samples, SEQ_LEN, use_revin_for_linear)
ols_pred = ols_pred_seq.reshape(test_samples.shape[0], -1)
linear_results.append(("Linear regression (OLS)", *evaluate_flat(ols_pred_seq, y_test_seq)))

ridge_alpha = 1.0
ridge = Ridge(alpha=ridge_alpha)
ridge_pred_seq = fit_predict_linear(ridge, train_samples, test_samples, SEQ_LEN, use_revin_for_linear)
ridge_pred = ridge_pred_seq.reshape(test_samples.shape[0], -1)
linear_results.append((f"Linear regression (Ridge alpha={ridge_alpha:g})", *evaluate_flat(ridge_pred_seq, y_test_seq)))

if "mse" in globals() and "mae" in globals():
    linear_results.insert(0, ("ARMD (notebook run)", float(mse), float(mae)))

print(f"train windows: {train_samples.shape}  test windows: {test_samples.shape}")
print(f"linear baseline RevIN: {use_revin_for_linear}  (matches ARMD.use_revin)")
print(f"{'model':<42}{'MSE':>12}{'MAE':>12}")
print("-" * 66)
for name, mse_i, mae_i in linear_results:
    print(f"{name:<42}{mse_i:>12.4f}{mae_i:>12.4f}")


### I-5  线性模型预测可视化

下面随机抽取几个测试窗口，把 OLS 的预测与真实未来放在同一张图上。若上一个 ARMD 评估 cell 已运行，也会同时画出 ARMD 的预测。


In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n_show = min(4, test_samples.shape[0])
idx_list = rng.choice(test_samples.shape[0], size=n_show, replace=False)
feat_show = min(2, test_samples.shape[-1])

ols_seq = ols_pred_seq
armd_available = "samples_last" in globals() and samples_last is not None

fig, axes = plt.subplots(n_show, feat_show, figsize=(6 * feat_show, 2.8 * n_show), sharex=True)
if n_show == 1:
    axes = np.array([axes])
if feat_show == 1:
    axes = axes[:, None]

for r, idx in enumerate(idx_list):
    for c in range(feat_show):
        ax = axes[r, c]
        xh = np.arange(SEQ_LEN)
        xf = np.arange(SEQ_LEN, 2 * SEQ_LEN)
        ax.plot(xh, test_samples[idx, :SEQ_LEN, c], color="#555", lw=0.9,
                label="history" if (r == 0 and c == 0) else None)
        ax.plot(xf, y_test_seq[idx, :, c], color="#1f77b4", lw=1.0,
                label="true future" if (r == 0 and c == 0) else None)
        ax.plot(xf, ols_seq[idx, :, c], color="#2ca02c", lw=1.1, ls="--",
                label="OLS" if (r == 0 and c == 0) else None)
        if armd_available:
            ax.plot(xf, samples_last[idx, :, c], color="#d62728", lw=1.1, ls=":",
                    label="ARMD" if (r == 0 and c == 0) else None)
        ax.axvline(SEQ_LEN - 0.5, color="grey", ls="--", lw=0.8)
        ax.grid(alpha=0.3)
        ax.set_title(f"test window #{int(idx)} feat{c}", fontsize=9)
        if c == 0:
            ax.set_ylabel("z-score", fontsize=8)

axes[0, 0].legend(loc="upper left", fontsize=8)
fig.suptitle("Linear baseline vs true future", y=1.01)
plt.tight_layout()
plt.show()


---
## Part J：公式索引 + 复现 Checklist

### J-1  论文全部公式索引（Eq.1–20）

| 公式编号 | 内容 | 代码位置 |
|---|---|---|
| **Eq.1** | 单步滑动 $X^t = \mathrm{Slide}(X^{t-1}, 1)$ | `Models/autoregressive_diffusion/armd.py::ARMD.q_sample`，`index=t_code+1` |
| **Eq.2** | t 步中间态 $= \sqrt{\bar\alpha_t}X^0 + \sqrt{1-\bar\alpha_t}z_t$ | `ARMD._train_loss` 使用 `q_sample` 得到 `x`，并用 `target_noise = (x - target*alpha)/minus_alpha` 反解该式 |
| **Eq.3** | 真实演化趋势 $z_t$ | `ARMD._train_loss`: `target_noise = (x - target*alpha)/minus_alpha` |
| **Eq.4** | 距离预测 $D = \mathrm{Linear}(X^t)$ | `Models/autoregressive_diffusion/linear.py::Linear.forward` → `x_tmp = linear(input_.T).T` |
| **Eq.5** | 预测 $\hat X^0$ | `Linear.forward`: `output = (alpha*input_ + (1-2*alpha)*x_tmp) / (1-alpha)^0.5` |
| **Eq.6** | 预测趋势 $\hat z$ | `ARMD.predict_noise_from_start(x_t, t, x0)` |
| **Eq.7** | L1 训练目标 $\mathcal{L}=\|z_t-\hat z\|$ | `ARMD._train_loss`: `loss_fn(pred_noise, target_noise)`，随后乘 `loss_weight` |
| **Eq.8** | DDIM 完整反向步（含 $\sigma_t\varepsilon_t$） | `ARMD.fast_sample`: 公式结构保留，但 `sigma = 0`、`noise = 0` 去掉随机项 |
| **Eq.9** | 确定性简化反向步（$\sigma_t=0$） | `ARMD.fast_sample`: `img = x_start * alpha_next.sqrt() + c * pred_noise` |
| **Eq.10** | 跳步加速采样 | `ARMD.fast_sample` 的 `time_pairs` 循环；`sampling_timesteps=2` 时 pairs 为 `(95,47),(47,-1)` |
| **Eq.11** | DDPM 单步前向 $q(X^t\|X^{t-1})$ | B-2b：ARMD 不执行高斯加噪；`ARMD.q_sample` 改为滑动切片 |
| **Eq.12** | DDPM 边缘 $q(X^t\|X^0)$ | B-2b/C-3：被 Eq.2/Eq.3 的确定性中间态分解替代 |
| **Eq.13** | $\bar\alpha_t = \prod_{k=1}^t \alpha_k$ | `ARMD.__init__`: `torch.cumprod(alphas, dim=0)` |
| **Eq.14** | 直接采样 $X^t = \sqrt{\bar\alpha}X^0 + \sqrt{1-\bar\alpha}\varepsilon$ | B-2b/C-3：只作类比；ARMD 用反解出的 `target_noise`，不是随机 $\varepsilon$ |
| **Eq.15** | DDPM 反向 $p_\theta(X^{t-1}\|X^t)$ | `ARMD.p_sample` 传统路径保留；Stock 复现走 `ARMD.fast_sample` Eq.8–10 |
| **Eq.16** | 条件 DDPM for TSF | B-2/B-2b：被替代方案；仓库没有条件编码器 `c=g(history)` |
| **Eq.17** | 条件单步去噪 | B-2/B-2b：被替代方案；历史直接作为采样起点 `x[:, :96, :]` |
| **Eq.18** | AR 成分 | B-3 动机类比；仓库没有单独 ARMA 模块 |
| **Eq.19** | MA 成分 | B-3 动机类比；趋势残差由 `target_noise`/`pred_noise` 承担 |
| **Eq.20** | 完整 ARMA 模型 | B-3 动机类比；仓库没有单独 ARMA 模块，可执行实现落在 Eq.1–10 |

### J-2  公式外但必须核对的代码实现

论文公式给出 ARMD 主干，但当前仓库还有几处会显著影响 Stock 复现结果的实现选择：

| 实现点 | 原始代码位置 | 本教程位置 | 为什么重要 |
|---|---|---|---|
| RevIN 窗口内归一化 | `ARMD.forward` / `ARMD.generate_mts` 的 `use_revin` 分支 | H-1/H-4/I-4 | Stock 趋势漂移明显；训练和推理都在每窗口历史统计量归一化空间进行，输出再反变换 |
| `q_sample` 的下标偏移 | `ARMD.q_sample`: `index = int(t[0]) + 1` | C-2/C-5/J checklist | 代码时间步 `t_code=0` 已经滑动 1 步，不是论文的 0 步初态 |
| 训练扰动的原地写入 | `Linear.forward`: `input_ += ...` | D-3/D-3b | `q_sample` 返回视图，训练扰动会影响同一 storage 上的 target 切片 |
| 双 schedule | `Linear.w` 用 linear；`ARMD` buffers 用 cosine | B-4/D-3/F-3 | `W(t)` 与 Eq.2/3/6 的 $\bar\alpha_t$ 不是同一组数 |
| 确定性采样 | `fast_sample`: `sigma = 0`, `noise = 0` | E-4/H-4 | Algorithm 2 的随机项在仓库实现中被关闭 |
| 采样裁剪/eta 无效 | `model_predictions`: `#x_start = maybe_clip(x_start)`；`fast_sample`: `sigma = 0` | E-4 | `clip_denoised=True` 不会裁剪 `x_start`；非零 `eta` 也会被覆盖为确定性采样 |
| EMA 推理 | `Trainer.sample_forecast`: `self.ema.ema_model.generate_mts(x)` | H-4/J checklist | 指标来自 EMA 模型，不是即时训练权重 |
| 测试 mask 不参与预测评分 | `Trainer.sample_forecast`: 读出 `t_m` 但后续不用；`reals = x[:, shape[0]:, :]` | G-3/H-4 | mask 标记未来目标区域，但 MSE/MAE 直接比较预测和 `x[:,96:,:]` |

### J-3  Algorithm 1/2 源码审计表

如果只看论文 Algorithm 1/2，很容易漏掉仓库实现中的 batch 时间步、RevIN、EMA、mask、确定性采样等细节。
下面这张表把“论文步骤 → 原始源码 → standalone 章节 → 实现备注”集中放在一起，作为复现前的最后审计入口。

| 论文算法步骤 | 原始源码证据 | standalone 位置 | 实现备注 |
|---|---|---|---|
| Algorithm 1 输入 $X^0_{1:T}$ | `Trainer.train`: `data = next(self.dl).to(device)`；`ARMD._train_loss`: `target = x_start[:,pred_len:,:]` | H-2 / F-4 | DataLoader 给的是完整 192 窗口；真实未来目标来自后 96 步 |
| Algorithm 1 采样时间步 $t$ | `ARMD.forward`: `torch.randint(0, self.num_timesteps, (1,)).repeat(b).long()` | D-5 / H-2 | 整个 batch 共享同一个 `t[0]`，不是每个样本独立时间步 |
| Algorithm 1 生成 $X^t$ | `ARMD.q_sample`: `index = int(t[0])+1`; `x_start[:,pred_len-index:-index,:]` | C-2 / C-5 | `t_code=0` 已滑动 1 步；`q_sample` 不使用传入的 `noise` |
| Algorithm 1 计算 $z_t$ | `_train_loss`: `target_noise = (x - target*alpha)/minus_alpha` | C-4 / D-5 | 不是随机高斯噪声，而是由确定性滑动状态反解出的演化趋势 |
| Algorithm 1 调用 $R(\cdot)$ | `ARMD.output` → `Linear.forward` → `self.linear(input_.permute(...))` | D-1 / F-3 | `Linear` 是 Devolution backbone；输入会先加训练扰动 |
| Algorithm 1 计算 $\hat z$ 和 loss | `_train_loss`: `pred_noise = ...`; `loss_fn(...); loss_weight` | D-4 / D-5b | Eq.7 之外还乘了代码级 `loss_weight` 时间步权重 |
| Algorithm 1 更新参数 | `Trainer.train`: `loss.backward`; `clip_grad_norm_`; `opt.step`; `sch.step`; `ema.update` | G-2 / H-2 | 训练循环还包含梯度累积、梯度裁剪、LR scheduler 和 EMA 更新 |
| Algorithm 2 输入历史序列 | `fast_sample`: `img = x[:,:pred_len,:]` | E-3 / H-4 | 采样起点是历史半段，不是随机噪声，也不是显式条件编码 |
| Algorithm 2 采样时间网格 | `torch.linspace(-1, total_timesteps - 1, steps=sampling_timesteps + 1)` | E-3 / E-4 | `sampling_timesteps=2` 表示 2 次网络更新；实际 pairs 为 `(95,47),(47,-1)` |
| Algorithm 2 调用训练后的 $R(\cdot)$ | `Trainer.sample_forecast`: `self.ema.ema_model.generate_mts(x)` | G-2 / H-4 | 评估使用 EMA 模型，不是即时训练权重 |
| Algorithm 2 Eq.10 更新 | `img = x_start * alpha_next.sqrt() + c * pred_noise` | E-3 / E-4 | `sigma=0; noise=0` 硬编码，采样过程确定性 |
| Algorithm 2 输出 $X^0_{1:T}$ | `if time_next < 0: img = x_start`; `return img` | E-4 / H-4 | `-1` 是结束哨兵；最后一步直接输出 $\hat X^0$ |
| Algorithm 2 评估真实值 | `sample_forecast`: `reals = x[:,shape[0]:,:]` | G-3 / H-4 | `mask` 被读出但不参与预测评分；MSE/MAE 对后 96 步逐元素计算 |
| Stock paper-style RevIN | `ARMD.forward` / `generate_mts`: `_revin_stats(x)` | H-1 / I-4 | 这是当前 `Config/stock_paper.yaml` 的关键协议字段，不在原论文算法伪代码中显式出现 |

### J-4  复现 Checklist

在与论文 Table 1 对比前，逐项确认：

- [ ] **数据**：`stock_data.csv` 来自 Diffusion-TS（6 列，无日期列），`name='stock'`
- [ ] **切分**：70/10/20 时间顺序（`three_split=True, train_ratio=0.7, val_ratio=0.1`）
- [ ] **归一化**：`StandardScaler.fit(全部行)`（不分段 fit）
- [ ] **Loss**：`loss_type='l1'`（Eq.7 是 L1，不是 L2）
- [ ] **采样步数**：`sampling_timesteps=2` 表示 2 次反向更新；实际时间网格由 `linspace(-1, T-1, steps=3)` 生成
- [ ] **RevIN**：`use_revin=True`，训练/推理都用 `x[:, :96, :]` 的窗口内统计量
- [ ] **批大小**：`batch_size=128`，`gradient_accumulate_every=2`
- [ ] **训练步数**：`max_epochs=2000`（`QUICK_TEST=False`）
- [ ] **训练时间步**：`torch.randint(..., (1,)).repeat(b)`，整 batch 共享同一个 `t[0]`
- [ ] **q_sample 偏移**：`index = t_code + 1`（不是 `t_code`）
- [ ] **确定性采样**：`fast_sample` 中 `sigma=0; noise=0`
- [ ] **推理起点**：`x[:, :96, :]`（历史半段，不是随机噪声）
- [ ] **EMA 推理**：`trainer.ema.ema_model`（不是 `trainer.model`）
- [ ] **指标空间**：z-score 归一化后（不反变换到原始价格）
- [ ] **10 次平均**：`for run in range(10)` 重复采样求均值
- [ ] **`w_grad=True`**：W(t) 可学习（不要固定为 alpha_bar_t）
